# Section 0 — Install & Imports

In [2]:
# Only run on Kaggle if packages are missing
!pip install pyyaml scikit-learn progress openslide-python h5py -q

In [3]:
#guars
run_mnist=False
luna_extraction=False

# Section 1 — utils/utils.py 

In [4]:
import sys
import math
import numpy as np
from sklearn.metrics import accuracy_score, roc_auc_score
from collections import defaultdict

import torch
from torch import nn

class Struct:
    def __init__(self, **entries):
        self.__dict__.update(entries)

def adjust_learning_rate(n_epoch_warmup, n_epoch, max_lr, optimizer, dloader, step):
    """
    Set learning rate according to cosine schedule
    """

    max_steps = int(n_epoch * len(dloader))
    warmup_steps = int(n_epoch_warmup * len(dloader))
    
    if step < warmup_steps:
        lr = max_lr * step / warmup_steps
    else:
        step -= warmup_steps
        max_steps -= warmup_steps
        q = 0.5 * (1 + math.cos(math.pi * step / max_steps))
        end_lr = max_lr * 0.001
        lr = max_lr * q + end_lr * (1 - q)

    optimizer.param_groups[0]['lr'] = lr

def shuffle_batch(x, shuffle_idx=None):
    """ shuffles each instance in batch the same way """
    
    if not torch.is_tensor(shuffle_idx):
        seq_len = x.shape[1]
        shuffle_idx = torch.randperm(seq_len)
    x = x[:, shuffle_idx]
    
    return x, shuffle_idx

def shuffle_instance(x, axis, shuffle_idx=None):
    """ shuffles each instance in batch in a different way """

    if not torch.is_tensor(shuffle_idx):
        # get permutation indices
        shuffle_idx = torch.rand(x.shape[:axis+1], device=x.device).argsort(axis)  
    
    idx_expand = shuffle_idx.clone().to(x.device)
    for _ in range(x.ndim-axis-1):
        idx_expand.unsqueeze_(-1)
    # reformat for gather operation
    idx_expand = idx_expand.repeat(*[1 for _ in range(axis+1)], *(x.shape[axis+1:]))  
    
    x = x.gather(axis, idx_expand)

    return x, shuffle_idx

class Logger(nn.Module):
    ''' Stores and computes statistiscs of losses and metrics '''

    def __init__(self, task_dict):
        super().__init__()

        self.task_dict = task_dict
        self.losses_it = defaultdict(list)
        self.losses_epoch = defaultdict(list)
        self.y_preds = defaultdict(list)
        self.y_trues = defaultdict(list)
        self.metrics = defaultdict(list)

    def update(self, next_loss, next_y_pred, next_y_true):
    
        for task in self.task_dict.values():
            t      = task['name']
            t_metr = task['metric']
            # Normalise : metric peut être str ou list
            if isinstance(t_metr, str):
                t_metr = [t_metr]
    
            self.losses_it[t].append(next_loss[t])
    
            # Décider comment stocker y_pred selon les métriques demandées
            has_auc_f1 = any(m in ['auc', 'f1', 'precision', 'recall']
                             for m in t_metr)
    
            if t_metr == ['accuracy']:
                # Seule accuracy → argmax suffit
                y_pred = np.argmax(next_y_pred[t], axis=-1)
            elif has_auc_f1:
                # AUC/F1/precision/recall ont besoin des scores bruts
                raw = next_y_pred[t]
                if raw.ndim == 2:
                    # softmax output [B, n_class] → garder proba classe positive (index 1)
                    y_pred = raw[:, 1].tolist()
                else:
                    # sigmoid output [B] → déjà un scalaire par sample
                    y_pred = raw.tolist()
            else:
                y_pred = next_y_pred[t].tolist()
    
            self.y_preds[t].extend(y_pred)
            self.y_trues[t].extend(next_y_true[t])

    def compute_metric(self):
    
        for task in self.task_dict.values():
            t = task['name']
            losses = self.losses_it[t]
            self.losses_epoch[t].append(np.mean(losses))
    
            # Collect all requested metrics (list or single string)
            requested = task['metric']
            if isinstance(requested, str):
                requested = [requested]
    
            y_true     = np.array(self.y_trues[t])
            y_pred_raw = np.array(self.y_preds[t])
    
            epoch_metrics = {}
            for current_metric in requested:
    
                if current_metric == 'accuracy':
                    # y_pred_raw peut être vecteurs [B, n_class] ou scalaires [B]
                    if y_pred_raw.ndim == 2:
                        y_pred = np.argmax(y_pred_raw, axis=-1)
                    else:
                        # scalaires = proba classe 1 → seuil 0.5
                        y_pred = (y_pred_raw >= 0.5).astype(int)
                    epoch_metrics[current_metric] = accuracy_score(
                        y_true, y_pred)
    
                elif current_metric == 'multilabel_accuracy':
                    y_pred = np.where(y_pred_raw >= 0.5, 1., 0.)
                    correct = np.all(y_pred == y_true, axis=-1).sum()
                    epoch_metrics[current_metric] = correct / len(y_true)
    
                elif current_metric == 'auc':
                    # robuste aux deux formats
                    scores = y_pred_raw[:, 1] \
                             if y_pred_raw.ndim == 2 else y_pred_raw
                    epoch_metrics[current_metric] = roc_auc_score(
                        y_true, scores)
    
                elif current_metric == 'f1':
                    if y_pred_raw.ndim == 2:
                        y_pred = np.argmax(y_pred_raw, axis=-1)
                    else:
                        y_pred = (y_pred_raw >= 0.5).astype(int)
                    from sklearn.metrics import f1_score
                    epoch_metrics[current_metric] = f1_score(
                        y_true, y_pred, zero_division=0)
    
                elif current_metric == 'precision':
                    if y_pred_raw.ndim == 2:
                        y_pred = np.argmax(y_pred_raw, axis=-1)
                    else:
                        y_pred = (y_pred_raw >= 0.5).astype(int)
                    from sklearn.metrics import precision_score
                    epoch_metrics[current_metric] = precision_score(
                        y_true, y_pred, zero_division=0)
    
                elif current_metric == 'recall':
                    if y_pred_raw.ndim == 2:
                        y_pred = np.argmax(y_pred_raw, axis=-1)
                    else:
                        y_pred = (y_pred_raw >= 0.5).astype(int)
                    from sklearn.metrics import recall_score
                    epoch_metrics[current_metric] = recall_score(
                        y_true, y_pred, zero_division=0)
    
            self.metrics[t].append(epoch_metrics)
    
            # reset
            self.losses_it[t] = []
            self.y_preds[t]   = []
            self.y_trues[t]   = []
    
    def print_stats(self, epoch, train, **kwargs):
    
        print_str  = 'Train' if train else 'Test'
        print_str += " Epoch: {}\n".format(epoch + 1)
    
        avg_loss = 0
        for task in self.task_dict.values():
            t         = task['name']
            mean_loss = self.losses_epoch[t][epoch]
            avg_loss += mean_loss
    
            epoch_metrics = self.metrics[t][epoch]
            metrics_str   = ', '.join(
                f"{k}: {v:.5f}" for k, v in epoch_metrics.items()
            )
            print_str += "task: {}, loss: {:.5f}, {}\n".format(
                t, mean_loss, metrics_str)
    
        avg_loss /= len(self.task_dict.values())
        print_str += "avg loss: {:.5f}".format(avg_loss)
    
        for k, v in kwargs.items():
            print_str += ", {}: {}".format(k, v)
        print_str += "\n"
    
        print(print_str)

# Model-related

## Section 2 — architecture/transformer.py 

In [5]:
import math

import torch
from torch import nn

def pos_enc_1d(D, len_seq):
    
    if D % 2 != 0:
        raise ValueError("Cannot use sin/cos positional encoding with "
                         "odd dim (got dim={:d})".format(D))
    pe = torch.zeros(len_seq, D)
    position = torch.arange(0, len_seq).unsqueeze(1)
    div_term = torch.exp((torch.arange(0, D, 2, dtype=torch.float) *
                         -(math.log(10000.0) / D)))
    pe[:, 0::2] = torch.sin(position.float() * div_term)
    pe[:, 1::2] = torch.cos(position.float() * div_term)

    return pe

class ScaledDotProductAttention(nn.Module):
    ''' Dot-Product Attention '''

    def __init__(self, temperature, attn_dropout=0.1):
        super().__init__()
        
        self.temperature = temperature
        self.dropout = nn.Dropout(attn_dropout)

    def compute_attn(self, q, k):
        
        attn = torch.matmul(q / self.temperature, k.transpose(2, 3))
        attn = self.dropout(torch.softmax(attn, dim=-1))

        return attn

    def forward(self, q, k, v):
        
        attn = self.compute_attn(q, k)
        output = torch.matmul(attn, v)

        return output

class MultiHeadCrossAttention(nn.Module):
    ''' Multi-head cross-attention module '''

    def __init__(self, n_token, H, D, D_k, D_v, attn_dropout=0.1, dropout=0.1):
        super().__init__()
        
        self.n_token = n_token
        self.H = H
        self.D_k = D_k
        self.D_v = D_v

        self.q = nn.Parameter(torch.empty((1, n_token, D)))
        q_init_val = math.sqrt(1 / D_k)
        nn.init.uniform_(self.q, a=-q_init_val, b=q_init_val)

        self.q_w = nn.Linear(D, H * D_k, bias=False)
        self.k_w = nn.Linear(D, H * D_k, bias=False)
        self.v_w = nn.Linear(D, H * D_v, bias=False)
        self.fc = nn.Linear(H * D_v, D, bias=False)

        self.attention = ScaledDotProductAttention(
            temperature=D_k ** 0.5,
            attn_dropout=attn_dropout
        )

        self.dropout = nn.Dropout(dropout)
        self.layer_norm = nn.LayerNorm(D, eps=1e-6)

    def get_attn(self, x):
        
        D_k, H, n_token = self.D_k, self.H, self.n_token
        B, len_seq = x.shape[:2]

        q = self.q_w(self.q).view(1, n_token, H, D_k)
        k = self.k_w(x).view(B, len_seq, H, D_k)

        q, k = q.transpose(1, 2), k.transpose(1, 2)

        attn = self.attention.compute_attn(q, k)

        return attn

    def forward(self, x):
        
        D_k, D_v, H, n_token = self.D_k, self.D_v, self.H, self.n_token
        B, len_seq = x.shape[:2]

        # project and separate heads
        q = self.q_w(self.q).view(1, n_token, H, D_k)
        k = self.k_w(x).view(B, len_seq, H, D_k)
        v = self.v_w(x).view(B, len_seq, H, D_v)

        # transpose for attention dot product: B x H x len_seq x D_k or D_v
        q, k, v = q.transpose(1, 2), k.transpose(1, 2), v.transpose(1, 2)
        # cross-attention
        x = self.attention(q, k, v)

        # transpose again: B x n_token x H x D_v
        # concat heads: B x n_token x (H * D_v)
        x = x.transpose(1, 2).contiguous().view(B, n_token, -1)
        # combine heads
        x = self.dropout(self.fc(x))
        # residual connection + layernorm
        x += self.q
        x = self.layer_norm(x)

        return x

class MLP(nn.Module):
    ''' MLP consisting of two feed-forward layers '''

    def __init__(self, D, D_inner, dropout=0.1):
        super().__init__()
        
        self.w_1 = nn.Linear(D, D_inner)
        self.w_2 = nn.Linear(D_inner, D)
        self.layer_norm = nn.LayerNorm(D, eps=1e-6)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):

        residual = x

        x = self.w_2(torch.relu(self.w_1(x)))
        x = self.dropout(x)
        
        x += residual
        x = self.layer_norm(x)

        return x

class Transformer(nn.Module):
    """ Cross-attention based transformer module """

    def __init__(self, n_token, H, D, D_k, D_v, D_inner, attn_dropout=0.1, dropout=0.1):
        super().__init__()
        
        self.crs_attn = MultiHeadCrossAttention(n_token, H, D, D_k, D_v, attn_dropout=attn_dropout, dropout=dropout)
        self.mlp = MLP(D, D_inner, dropout=dropout)
    
    def get_scores(self, x):

        attn = self.crs_attn.get_attn(x)
        # Average scores over heads and tasks
        # Average over tasks is only required for multi-task learning (mnist).
        return attn.mean(dim=1).transpose(1, 2).mean(-1)

    def forward(self, x):

        return self.mlp(self.crs_attn(x))

## Section 3 — architecture/ips_net.py 

In [6]:
import sys
import math

import torch
import torch.nn as nn
from torchvision.models import resnet18, resnet50, ResNet18_Weights, ResNet50_Weights


# from utils.utils import shuffle_batch, shuffle_instance
# from architecture.transformer import Transformer, pos_enc_1d


class IPSNet(nn.Module):
    """
    Net that runs all the main components:
    patch encoder, IPS, patch aggregator and classification head
    """

    def get_conv_patch_enc(self, enc_type, pretrained, n_chan_in, n_res_blocks):
        # Get architecture for patch encoder
        if enc_type == 'resnet18': 
            res_net_fn = resnet18
            weights=ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
        elif enc_type == 'resnet50':
            res_net_fn = resnet50
            # Resnet50 pretrained weights not used in experiments
            weights=ResNet50_Weights.IMAGENET1K_V1 if pretrained else None        

        res_net = res_net_fn(weights=weights)

        if n_chan_in == 1:
            # Standard resnet uses 3 input channels
            res_net.conv1 = nn.Conv2d(n_chan_in, 64, kernel_size=7, stride=2, padding=3, bias=False)
        
        # Compose patch encoder
        layer_ls = []
        layer_ls.extend([
            res_net.conv1,
            res_net.bn1,
            res_net.relu,
            res_net.maxpool,
            res_net.layer1,
            res_net.layer2
        ])

        if n_res_blocks == 4:
            layer_ls.extend([
                res_net.layer3,
                res_net.layer4
            ])
        
        layer_ls.append(res_net.avgpool)

        return nn.Sequential(*layer_ls)

    def get_projector(self, n_chan_in, D):
        return nn.Sequential(
            nn.LayerNorm(n_chan_in, eps=1e-05, elementwise_affine=False),
            nn.Linear(n_chan_in, D),
            nn.BatchNorm1d(D),
            nn.ReLU()
        )   

    def get_output_layers(self, tasks):
        """
        Create an output layer for each task according to task definition
        """

        D = self.D
        n_class = self.n_class

        output_layers = nn.ModuleDict()
        for task in tasks.values():
            if task['act_fn'] == 'softmax':
                act_fn = nn.Softmax(dim=-1)
            elif task['act_fn'] == 'sigmoid':
                act_fn = nn.Sigmoid()
            
            layers = [
                nn.Linear(D, n_class),
                act_fn
            ]
            output_layers[task['name']] = nn.Sequential(*layers)

        return output_layers

    def __init__(self, device, conf):
        super().__init__()

        self.device = device
        self.n_class = conf.n_class
        self.M = conf.M
        self.I = conf.I
        self.D = conf.D 
        self.use_pos = conf.use_pos
        self.tasks = conf.tasks
        self.shuffle = conf.shuffle
        self.shuffle_style = conf.shuffle_style
        self.is_image = conf.is_image

        if self.is_image:
            self.encoder = self.get_conv_patch_enc(conf.enc_type, conf.pretrained,
                conf.n_chan_in, conf.n_res_blocks)
        else:
            self.encoder = self.get_projector(conf.n_chan_in, self.D)

        # Define the multi-head cross-attention transformer
        self.transf = Transformer(conf.n_token, conf.H, conf.D, conf.D_k, conf.D_v,
            conf.D_inner, conf.attn_dropout, conf.dropout)

        # Optionally use standard 1d sinusoidal positional encoding
        if conf.use_pos:
            self.pos_enc = pos_enc_1d(conf.D, conf.N).unsqueeze(0).to(device)
        else:
            self.pos_enc = None
        
        # Define an output layer for each task
        self.output_layers = self.get_output_layers(conf.tasks)

    def do_shuffle(self, patches, pos_enc):
        """
        Shuffles patches and pos_enc so that patches that have an equivalent score
        are sampled uniformly
        """

        shuffle_style = self.shuffle_style
        if shuffle_style == 'batch':
            patches, shuffle_idx = shuffle_batch(patches)
            if torch.is_tensor(pos_enc):
                pos_enc, _ = shuffle_batch(pos_enc, shuffle_idx)
        elif shuffle_style == 'instance':
            patches, shuffle_idx = shuffle_instance(patches, 1)
            if torch.is_tensor(pos_enc):
                pos_enc, _ = shuffle_instance(pos_enc, 1, shuffle_idx)
        
        return patches, pos_enc

    def score_and_select(self, emb, emb_pos, M, idx):
        """ 
        Scores embeddings and selects the top-M embeddings
        """
        D = emb.shape[2]

        emb_to_score = emb_pos if torch.is_tensor(emb_pos) else emb

        # Obtain scores from transformer
        attn = self.transf.get_scores(emb_to_score) # (B, M+I)

        # Get indixes of top-scoring patches
        top_idx = torch.topk(attn, M, dim = -1)[1] # (B, M)
        
        # Update memory buffers
        # Note: Scoring is based on `emb_to_score`, selection is based on `emb`
        mem_emb = torch.gather(emb, 1, top_idx.unsqueeze(-1).expand(-1,-1,D))
        mem_idx = torch.gather(idx, 1, top_idx)

        return mem_emb, mem_idx

    def get_preds(self, embeddings):
        preds = {}
        for task in self.tasks.values():
            t_name, t_id = task['name'], task['id']
            layer = self.output_layers[t_name]

            emb = embeddings[:,t_id]
            preds[t_name] = layer(emb)            

        return preds

    # IPS runs in no-gradient mode
    @torch.no_grad()
    def ips(self, patches):
        """ Iterative Patch Selection """

        # Get useful variables
        M = self.M
        I = self.I
        D = self.D  
        device = self.device
        shuffle = self.shuffle
        use_pos = self.use_pos
        pos_enc = self.pos_enc
        patch_shape = patches.shape
        B, N = patch_shape[:2]

        # Shortcut: IPS not required when memory is larger than total number of patches
        if M >= N:
            # Batchify pos enc
            pos_enc = pos_enc.expand(B, -1, -1) if use_pos else None
            return patches.to(device), pos_enc 

        # IPS runs in evaluation mode
        if self.training:
            self.encoder.eval()
            self.transf.eval()

        # Batchify positional encoding
        if use_pos:
            pos_enc = pos_enc.expand(B, -1, -1)

        # Shuffle patches (i.e., randomize when patches obtain identical scores)
        if shuffle:
            patches, pos_enc = self.do_shuffle(patches, pos_enc)

        # Init memory buffer
        # Put patches onto GPU in case it is not there yet (lazy loading).
        # `to` will return self in case patches are located on GPU already (eager loading)
        init_patch = patches[:,:M].to(device) 
        
        ## Embed
        mem_emb = self.encoder(init_patch.reshape(-1, *patch_shape[2:]))
        mem_emb = mem_emb.view(B, M, -1)
        
        # Init memory indixes in order to select patches at the end of IPS.
        idx = torch.arange(N, dtype=torch.int64, device=device).unsqueeze(0).expand(B, -1)
        mem_idx = idx[:,:M]

        # Apply IPS for `n_iter` iterations
        n_iter = math.ceil((N - M) / I)
        for i in range(n_iter):
            # Get next patches
            start_idx = i * I + M
            end_idx = min(start_idx + I, N)

            iter_patch = patches[:, start_idx:end_idx].to(device)
            iter_idx = idx[:, start_idx:end_idx]

            # Embed
            iter_emb = self.encoder(iter_patch.reshape(-1, *patch_shape[2:]))
            iter_emb = iter_emb.view(B, -1, D)
            
            # Concatenate with memory buffer
            all_emb = torch.cat((mem_emb, iter_emb), dim=1)
            all_idx = torch.cat((mem_idx, iter_idx), dim=1)
            # When using positional encoding, also apply it during patch selection
            if use_pos:
                all_pos_enc = torch.gather(pos_enc, 1, all_idx.view(B, -1, 1).expand(-1, -1, D))
                all_emb_pos = all_emb + all_pos_enc
            else:
                all_emb_pos = None

            # Select Top-M patches according to cross-attention scores
            mem_emb, mem_idx = self.score_and_select(all_emb, all_emb_pos, M, all_idx)

        # Select patches
        n_dim_expand = len(patch_shape) - 2
        mem_patch = torch.gather(patches, 1, 
            mem_idx.view(B, -1, *(1,)*n_dim_expand).expand(-1, -1, *patch_shape[2:]).to(patches.device)
        ).to(device)

        if use_pos:
            mem_pos = torch.gather(pos_enc, 1, mem_idx.unsqueeze(-1).expand(-1, -1, D))
        else:
            mem_pos = None

        # Set components back to training mode
        # Although components of `self` that are relevant for IPS have been set to eval mode,
        # self is still in training mode at training time, i.e., we can use it here.
        if self.training:
            self.encoder.train()
            self.transf.train()
    
        # Return selected patch and corresponding positional embeddings
        return mem_patch, mem_pos

    def forward(self, mem_patch, mem_pos=None):
        """
        After M patches have been selected during IPS, encode and aggregate them.
        The aggregated embedding is input to a classification head.
        """

        patch_shape = mem_patch.shape
        B, M = patch_shape[:2]

        mem_emb = self.encoder(mem_patch.reshape(-1, *patch_shape[2:]))
        mem_emb = mem_emb.view(B, M, -1)        

        if torch.is_tensor(mem_pos):
            mem_emb = mem_emb + mem_pos

        image_emb = self.transf(mem_emb)

        preds = self.get_preds(image_emb)
        
        return preds

# Section 4 — Dataset cells (3 sub-sections)

## create mnist

In [7]:
if run_mnist:
        # Adapted from https://github.com/idiap/attention-sampling
    
    import os
    import argparse
    import json
    
    import numpy as np
    from keras.datasets import mnist
    
    class MegapixelMNIST:
        """
        Class to create an artificial megapixel mnist dataset
        """
    
        class Sample(object):
            def __init__(self, dataset, idxs, positions, noise_positions,
                         noise_patterns):
                self._dataset = dataset
                self._idxs = idxs
                self._positions = positions
                self._noise_positions = noise_positions
                self._noise_patterns = noise_patterns
    
                self._img = None
    
            @property
            def noise_positions_and_patterns(self):
                return zip(self._noise_positions, self._noise_patterns)
    
            def _get_slice(self, pos, s=28, scale=1, offset=(0, 0)):
                pos = (int(pos[0]*scale-offset[0]), int(pos[1]*scale-offset[1]))
                s = int(s)
                return (
                    slice(max(0, pos[0]), max(0, pos[0]+s)),
                    slice(max(0, pos[1]), max(0, pos[1]+s)),
                    0
                )
    
            def create_img(self):
                if self._img is None:
                    size = self._dataset._H, self._dataset._W
                    img = np.zeros(size + (1,), dtype=np.uint8)
                    
                    if self._dataset._should_add_noise:
                        for p, i in self.noise_positions_and_patterns:
                            img[self._get_slice(p)] = \
                                255*self._dataset._noise[i]
                    
                    for p, i in zip(self._positions, self._idxs):
                        img[self._get_slice(p)] = \
                            255*self._dataset._images[i]
                    
                    self._img = img
                return self._img
    
        def __init__(self, N=5000, W=1500, H=1500, train=True,
                     noise=True, n_noise=50, seed=0):
            # Load the images
            x, y = mnist.load_data()[0 if train else 1]
            x = x.astype(np.float32) / 255.
    
            self._W, self._H = W, H
            self._images = x
    
            # Generate the dataset
            try:
                random_state = np.random.get_state()
                np.random.seed(seed + int(train))
                self._nums, self._targets, self._digits, self._max_targets = self._get_numbers(N, y)
                self._pos = self._get_positions(N, W, H)
                self._top_targets = self._get_top_targets()
    
                self._noise, self._noise_positions, self._noise_patterns = \
                    self._create_noise(N, W, H, n_noise)
            finally:
                np.random.set_state(random_state)
    
            # Boolean whether to add noise
            self._should_add_noise = noise
    
        def _create_noise(self, N, W, H, n_noise):
            """
            Create some random scribble noise of straight lines
            """
            angles = np.tan(np.random.rand(n_noise)*np.pi/2.5)
            A = np.zeros((n_noise, 28, 28))
            for i in range(n_noise):
                m = min(27.49, 27.49/angles[i])
                x = np.linspace(0, m, 56)
                y = angles[i]*x
                A[i, np.round(x).astype(int), np.round(y).astype(int)] = 1.
            B = np.array(A)
            np.random.shuffle(B)
            flip_x = np.random.rand(n_noise) < 0.33
            flip_y = np.random.rand(n_noise) < 0.33
            B[flip_x] = np.flip(B[flip_x], 2)
            B[flip_y] = np.flip(B[flip_y], 2)
            noise = ((A + B) > 0).astype(float)
            noise *= np.random.rand(n_noise, 28, 28)*0.2 + 0.8
            noise = noise.astype(np.float32)
    
            # Randomly assign noise to all images
            positions = (np.random.rand(N, n_noise, 2)*[H-56, W-56] + 28).astype(int)
            patterns = (np.random.rand(N, n_noise)*n_noise).astype(int)
    
            return noise, positions, patterns
    
        def _get_numbers(self, N, y):
            """
            Method to get numbers from the dataset
    
            Parameters:
            N (int): Number of samples (megapixel images) to create
            y (numpy array): Labels of standard mnist images
    
            Returns:
            numpy array: Array of indexes of the selected samples
            numpy array: Array of targets for the selected samples (task majority)
            numpy array: Array of digits for the selected samples (task multilabel)
            numpy array: Array of maximum digits for the selected samples (task max)
            """
            # Initialize empty lists for the output arrays
            nums = []
            targets = []
            max_targets = []
            all_digits = []
            # Get all indexes of the dataset
            all_idxs = np.arange(len(y))
            # Loop over the required number of samples
            for _ in range(N):
                # Get a random digit
                target = int(np.random.rand()*10)
                # Get three indexes where the target digit is present
                positive_idxs = np.random.choice(all_idxs[y == target], 3)
                # Get two negative indexes where the target digit is not present
                neg_idxs = np.random.choice(all_idxs[y != target], 2)
    
                # Concatenate the positive and negative indexes
                pos_neg_idxs = np.concatenate([positive_idxs, neg_idxs])
                # Get the digits from the concatenated indexes
                digits = y[pos_neg_idxs]
                # Get the maximum digit from the digits array
                max_target = np.max(digits)
    
                # Append the outputs to their respective lists
                nums.append(pos_neg_idxs)
                targets.append(target)
                all_digits.append(digits)
                max_targets.append(max_target)
    
            # Convert the lists to numpy arrays and return
            return np.array(nums), np.array(targets), np.array(all_digits), np.array(max_targets)
    
        def _get_positions(self, N, W, H):
            """
            Generates random positions of 5 digits in an image
            with size (H, W)
    
            Arguments:
                N: number of images to generate positions for
                W: width of the image
                H: height of the image
    
            Returns:
                np.array with shape (N, 5, 2) containing positions of 5 digits
            """
            def overlap(positions, pos):
                """
                Check if the new position 'pos' overlaps with
                any of the existing positions in 'positions'
                """
                if len(positions) == 0:
                    return False
                distances = np.abs(
                    np.asarray(positions) - np.asarray(pos)[np.newaxis]
                )
                axis_overlap = distances < 28
                return np.logical_and(axis_overlap[:, 0], axis_overlap[:, 1]).any()
    
            positions = []
            for _ in range(N):
                position = []
                for _ in range(5):
                    while True:
                        pos = np.round(np.random.rand(2)*[H-28, W-28]).astype(int)
                        if not overlap(position, pos):
                            break
                    position.append(pos)
                positions.append(position)
    
            return np.array(positions)
        
        def _get_top_targets(self):
            """
            Get digit that is topmost in each image
            """
            # `pos` is a numpy array with shape (n_img, digits, height and width)
            pos_height = self._pos[:,:,0] 
            
            # Get the index of the digit with the minimum height, i.e. top-most
            top_pos_idx = np.argmin(pos_height, axis=-1)
            
            # Get the digit with the minimum height for each image
            N = self._digits.shape[0]
            top_targets = self._digits[np.arange(N), top_pos_idx]
    
            return top_targets
    
        def __len__(self):
            return len(self._nums)
    
        def __getitem__(self, i):
            if len(self) <= i:
                raise IndexError()
            # Create a new sample
            sample = self.Sample(
                self,
                self._nums[i],
                self._pos[i],
                self._noise_positions[i],
                self._noise_patterns[i]
            )
            x = sample.create_img().astype(np.float32) / 255
            # Obtain labels for all tasks
            y = self._targets[i]
            y_max = self._max_targets[i]
            y_top = self._top_targets[i]
            y_multi = np.eye(10)[self._digits[i]].sum(0).clip(0,1)
    
            return x, y, y_max, y_top, y_multi
    
    
    def sparsify(dataset):
        """
        Store non-zero values and their indixes only to save memory
        """
        def to_sparse(x):
            x = x.ravel()
            indices = np.where(x != 0)
            values = x[indices]
            return (indices, values)
    
        print("Sparsifying dataset")
        data = []
        for i, (x, y_maj, y_max, y_top, y_multi) in enumerate(dataset):
            print(
                "\u001b[1000DProcessing {:5d} /  {:5d}".format(i+1, len(dataset)),
                end="",
                flush=True
            )
    
            data.append({
                'input': to_sparse(x),
                'majority': y_maj,
                'max': y_max,
                'top': y_top,
                'multi': y_multi
            })
        print()
        return data
    
    def main(argv):
        parser = argparse.ArgumentParser(
            description="Create the Megapixel MNIST dataset"
        )
        parser.add_argument(
            "--n_train",
            type=int,
            default=5000,
            help="How many images to create for training set"
        )
        parser.add_argument(
            "--n_test",
            type=int,
            default=1000,
            help="How many images to create for test set"
        )
        parser.add_argument(
            "--width",
            type=int,
            default=1500,
            help="Set the width for the image"
        )
        parser.add_argument(
            "--height",
            type=int,
            default=1500,
            help="Set the height for the image"
        )
        parser.add_argument(
            "--no_noise",
            action="store_false",
            dest="noise",
            help="Do not use noise in the dataset"
        )
        parser.add_argument(
            "--n_noise",
            type=int,
            default=50,
            help="Set the number of noise patterns per image"
        )
        parser.add_argument(
            "--dataset_seed",
            type=int,
            default=0,
            help="Choose the random seed for the dataset"
        )
        parser.add_argument(
            "output_directory",
            help="The directory to save the dataset into"
        )
    
        args = parser.parse_args(argv)
    
        if not os.path.exists(args.output_directory):
            os.makedirs(args.output_directory)
    
        with open(os.path.join(args.output_directory, "parameters.json"), "w") as f:
            json.dump(
                {
                    "n_train": args.n_train,
                    "n_test": args.n_test,
                    "width": args.width,
                    "height": args.height,
                    "noise": args.noise,
                    "n_noise": args.n_noise,
                    "seed": args.dataset_seed
                },
                f,
                indent=4
            )
        
        # Write the training set
        training = MegapixelMNIST(
            N=args.n_train,
            train=True,
            W=args.width,
            H=args.height,
            noise=args.noise,
            n_noise=args.n_noise,
            seed=args.dataset_seed
        )
        data = sparsify(training)
        np.save(os.path.join(args.output_directory, "train.npy"), data)
    
        # Write the test set
        test = MegapixelMNIST(
            N=args.n_test,
            train=False,
            W=args.width,
            H=args.height,
            noise=args.noise,
            n_noise=args.n_noise,
            seed=args.dataset_seed
        )
        data = sparsify(test)
        np.save(os.path.join(args.output_directory, "test.npy"), data)
    
    # Usage example: python make_mnist.py --width 1500 --height 1500 dsets/megapixel_mnist_1500
    # if __name__ == "__main__":
    #     main(None)

    pass

## run mnist

In [8]:
# Run this cell once to generate the Megapixel MNIST dataset

if run_mnist:
    main([
    '--width', '1500',
    '--height', '1500',
    '--n_train', '5000',
    '--n_test', '1000',
    '/kaggle/working/dsets/megapixel_mnist_1500'  # ← your output path
])
    pass


## 4a — data/megapixel_mnist/make_mnist.py


In [9]:
import os
import json
import numpy as np
import torch

class MegapixelMNIST(torch.utils.data.Dataset):
    """ Loads the Megapixel MNIST dataset """

    def __init__(self, conf, train=True):
        with open(os.path.join(conf.data_dir, "parameters.json")) as f:
            self.parameters = json.load(f)

        self.patch_size = conf.patch_size
        self.patch_stride = conf.patch_stride
        self.tasks = conf.tasks

        filename = "train.npy" if train else "test.npy"
        W = self.parameters["width"]
        H = self.parameters["height"]

        self._img_shape = (H, W, 1)
        self._data = np.load(os.path.join(conf.data_dir, filename), allow_pickle=True)

    def __len__(self):
        return len(self._data)

    def __getitem__(self, i):
        if i >= len(self):
            raise IndexError()

        patch_size = self.patch_size
        patch_stride = self.patch_stride

        # Placeholders
        img = np.zeros(self._img_shape, dtype=np.float32).ravel()

        # Fill the sparse representations
        data = self._data[i]
        img[data['input'][0]] = data['input'][1]

        # Reshape to final shape        
        img = img.reshape(self._img_shape)
        img = torch.from_numpy(img)
        img = img.permute(2, 0, 1)

        # Extract patches
        patches = img.unfold(
            1, patch_size[0], patch_stride[0]
        ).unfold(
            2, patch_size[1], patch_stride[1]
        ).permute(1, 2, 0, 3, 4)
        
        patches = patches.reshape(-1, *patches.shape[2:])

        data_dict = {'input': patches}
        for task in self.tasks.values():
            data_dict[task['name']] = data[task['name']] 

        return data_dict

# Traffic Sign

## 4b — data/traffic/traffic_dataset.py
Copy

In [10]:
import os
from os import path
import sys
import hashlib
from functools import partial
from collections import namedtuple
import urllib.request
import zipfile
from PIL import Image

from torch.utils.data import Dataset
from torchvision import transforms

import ssl
ssl._create_default_https_context = ssl._create_unverified_context

### Adapted from: https://github.com/sara-nl/attention-sampling-pytorch

def check_file(filepath, md5sum):
    """Check a file against an md5 hash value.
    Returns
    -------
        True if the file exists and has the given md5 sum False otherwise
    """

    try:
        md5 = hashlib.md5()
        with open(filepath, "rb") as f:
            for chunk in iter(partial(f.read, 4096), b""):
                md5.update(chunk)
        return md5.hexdigest() == md5sum
    except FileNotFoundError:
        return False

def ensure_dataset_exists(directory, tries=1, progress_file=sys.stderr):
    """Ensure that the dataset is downloaded and is correct.
    Correctness is checked only against the annotations files.
    """

    set1_url = ("http://www.isy.liu.se/cvl/research/trafficSigns"
                "/swedishSignsSummer/Set1/Set1Part0.zip")
    set1_annotations_url = ("http://www.isy.liu.se/cvl/research/trafficSigns"
                            "/swedishSignsSummer/Set1/annotations.txt")
    set1_annotations_md5 = "9106a905a86209c95dc9b51d12f520d6"
    set2_url = ("http://www.isy.liu.se/cvl/research/trafficSigns"
                "/swedishSignsSummer/Set2/Set2Part0.zip")
    set2_annotations_url = ("http://www.isy.liu.se/cvl/research/trafficSigns"
                            "/swedishSignsSummer/Set2/annotations.txt")
    set2_annotations_md5 = "09debbc67f6cd89c1e2a2688ad1d03ca"

    integrity = (
        check_file(
            path.join(directory, "Set1", "annotations.txt"),
            set1_annotations_md5
        ) and check_file(
            path.join(directory, "Set2", "annotations.txt"),
            set2_annotations_md5
        )
    )

    if integrity:
        return

    if tries <= 0:
        raise RuntimeError(("Cannot download dataset or dataset download "
                            "is corrupted"))

    print("Downloading Set1", file=progress_file)
    download_file(set1_url, path.join(directory, "Set1.zip"),
                  progress_file=progress_file)
    print("Extracting...", file=progress_file)
    with zipfile.ZipFile(path.join(directory, "Set1.zip")) as archive:
        archive.extractall(path.join(directory, "Set1"))
    print("Getting annotation file", file=progress_file)
    download_file(
        set1_annotations_url,
        path.join(directory, "Set1", "annotations.txt"),
        progress_file=progress_file
    )
    print("Downloading Set2", file=progress_file)
    download_file(set2_url, path.join(directory, "Set2.zip"),
                  progress_file=progress_file)
    print("Extracting...", file=progress_file)
    with zipfile.ZipFile(path.join(directory, "Set2.zip")) as archive:
        archive.extractall(path.join(directory, "Set2"))
    print("Getting annotation file", file=progress_file)
    download_file(
        set2_annotations_url,
        path.join(directory, "Set2", "annotations.txt"),
        progress_file=progress_file
    )

    return ensure_dataset_exists(
        directory,
        tries=tries - 1,
        progress_file=progress_file
    )

def download_file(url, destination, progress_file=sys.stderr):
    """Download a file with progress."""
    
    response = urllib.request.urlopen(url)
    n_bytes = response.headers.get("Content-Length")
    if n_bytes == "":
        n_bytes = 0
    else:
        n_bytes = int(n_bytes)

    message = "\rReceived {} / {}"
    cnt = 0
    with open(destination, "wb") as dst:
        while True:
            print(message.format(cnt, n_bytes), file=progress_file,
                  end="", flush=True)
            data = response.read(65535)
            if len(data) == 0:
                break
            dst.write(data)
            cnt += len(data)
    print(file=progress_file)

class Sign(namedtuple("Sign", ["visibility", "bbox", "type", "name"])):
    """A sign object. Useful for making ground truth images as well as making
    the dataset."""

    @property
    def x_min(self):
        
        return self.bbox[2]

    @property
    def x_max(self):
        
        return self.bbox[0]

    @property
    def y_min(self):
        
        return self.bbox[3]

    @property
    def y_max(self):
        
        return self.bbox[1]

    @property
    def area(self):

        return (self.x_max - self.x_min) * (self.y_max - self.y_min)

    @property
    def center(self):

        return [
            (self.y_max - self.y_min) / 2 + self.y_min,
            (self.x_max - self.x_min) / 2 + self.x_min
        ]

    @property
    def visibility_index(self):

        visibilities = ["VISIBLE", "BLURRED", "SIDE_ROAD", "OCCLUDED"]
        return visibilities.index(self.visibility)

    def pixels(self, scale, size):

        return zip(*(
            (i, j)
            for i in range(round(self.y_min * scale), round(self.y_max * scale) + 1)
            for j in range(round(self.x_min * scale), round(self.x_max * scale) + 1)
            if i < round(size[0] * scale) and j < round(size[1] * scale)
        ))

    def __lt__(self, other):

        if not isinstance(other, Sign):
            raise ValueError("Signs can only be compared to signs")

        if self.visibility_index != other.visibility_index:
            return self.visibility_index < other.visibility_index

        return self.area > other.area


class STS:
    """The STS class reads the annotations and creates the corresponding
    Sign objects."""

    def __init__(self, directory, train=True, seed=0):

        cwd = os.getcwd().replace('dataset', '')
        directory = path.join(cwd, directory)
        ensure_dataset_exists(directory)

        self._directory = directory
        self._inner = "Set{}".format(1 + ((seed + 1 + int(train)) % 2))
        self._data = self._load_signs(self._directory, self._inner)

    def _load_files(self, directory, inner):

        files = set()
        with open(path.join(directory, inner, "annotations.txt")) as f:
            for l in f:
                files.add(l.split(":", 1)[0])

        return sorted(files)

    def _read_bbox(self, parts):

        def _float(x):

            try:
                return float(x)
            except ValueError:
                if len(x) > 0:
                    return _float(x[:-1])
                raise

        return [_float(x) for x in parts]

    def _load_signs(self, directory, inner):

        with open(path.join(directory, inner, "annotations.txt")) as f:
            lines = [l.strip() for l in f]
        keys, values = zip(*(l.split(":", 1) for l in lines))
        all_signs = []
        for v in values:
            signs = []
            for sign in v.split(";"):
                if sign == [""] or sign == "":
                    continue
                parts = [s.strip() for s in sign.split(",")]
                if parts[0] == "MISC_SIGNS":
                    continue
                signs.append(Sign(
                    visibility=parts[0],
                    bbox=self._read_bbox(parts[1:5]),
                    type=parts[5],
                    name=parts[6]
                ))
            all_signs.append(signs)
        images = [path.join(directory, inner, f) for f in keys]

        return list(zip(images, all_signs))

    def __len__(self):
        return len(self._data)

    def __getitem__(self, i):
        return self._data[i]

class TrafficSigns(Dataset):
    """ Loads images from the traffic signs dataset as
    a filtered version of the STS dataset.
    Arguments
    ---------
        directory: str, The directory that the dataset already is or is going
                   to be downloaded in
        train: bool, Select the training or testing sets
        seed: int, The prng seed for the dataset
    """

    LIMITS = ["50_SIGN", "70_SIGN", "80_SIGN"]
    CLASSES = ["EMPTY", *LIMITS]
    IMG_SIZE = (1200, 1600)

    def __init__(self, conf, train=True):

        self.patch_size = conf.patch_size
        self.patch_stride = conf.patch_stride
        self.tasks = conf.tasks
        
        self._data = self._filter(STS(conf.data_dir, train, conf.seed))
        
        transform_list = [
            transforms.Resize([*self.IMG_SIZE])
        ]

        if train:
            transform_list += [
                transforms.ColorJitter(0.1, 0.1, 0.1, 0.1),
                transforms.RandomAffine(degrees=0, translate=(100 / self.IMG_SIZE[1], 100 / self.IMG_SIZE[0])),
            ]
        
        transform_list += [
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ]
           
        self.transform = transforms.Compose(transform_list)

    def _filter(self, data):

        filtered = []
        for image, signs in data:
            signs, acceptable = self._acceptable(signs)
            if acceptable:
                if not signs:
                    filtered.append((image, 0))
                else:
                    filtered.append((image, self.CLASSES.index(signs[0].name)))
        return filtered

    def _acceptable(self, signs):

        # Keep it as empty
        if not signs:
            return signs, True

        # Filter just the speed limits and sort them wrt visibility
        signs = sorted(s for s in signs if s.name in self.LIMITS)

        # No speed limit but many other signs
        if not signs:
            return None, False

        # Not visible sign so skip
        if signs[0].visibility != "VISIBLE":
            return None, False

        return signs, True

    def __len__(self):

        return len(self._data)

    def __getitem__(self, i):

        patch_size = self.patch_size
        patch_stride = self.patch_stride

        img, category = self._data[i]
        img = Image.open(img)
        img = self.transform(img)

        # Extract patches
        patches = img.unfold(
            1, patch_size[0], patch_stride[0]
        ).unfold(
            2, patch_size[1], patch_stride[1]
        ).permute(1, 2, 0, 3, 4)

        patches = patches.reshape(-1, *patches.shape[2:])

        data_dict = {'input': patches}
        for task in self.tasks.values():
            data_dict[task['name']] = category

        return data_dict

# Camelyon

 ## data/camelyon/cam_utils.py 

In [11]:
import os
import fnmatch
from collections import namedtuple
from typing import Dict

from PIL import Image
from PIL import ImageDraw
from progress.bar import IncrementalBar

Point = namedtuple('Point', 'x y')

def find_files(pattern, path) -> Dict[str, str]:
    """
    Find files in a directory by given file name pattern.

    Parameters
    ----------
    pattern : str
        File pattern allowing wildcards.

    path : str
        Root directory to search in.

    Returns
    -------
    dict(str: str)
        Dictionary of all found files where the file names are keys and the relative paths
        from search root are values.
    """
    result = {}
    for root, dirs, files in os.walk(path):
        for name in files:
            if fnmatch.fnmatch(name, pattern):
                result[name] = os.path.join(root, name)
    return result

class ProgressBar(IncrementalBar):
    @property
    def remaining_fmt(self):
        m, s = divmod(self.eta, 60)
        h, m = divmod(m, 60)
        return f'{h:02}:{m:02}:{s:02}'

    @property
    def elapsed_fmt(self):
        m, s = divmod(self.elapsed, 60)
        h, m = divmod(m, 60)
        return f'{h:02}:{m:02}:{s:02}'

def draw_polygon(image: Image.Image, polygon, *, fill, outline) -> Image.Image:
    """
    Draw a filled polygon on to an image.

    Parameters
    ----------
    image : Image.Image
        Background image to be drawn on.

    polygon :
        Polygon to be drawn.

    fill : color str or tuple
        Fill color.

    outline : color str or tuple
        Outline color.

    Returns
    -------
    Image.Image
        A copy of the background image with the polygon drawn onto.
    """
    img_back = image
    img_poly = Image.new('RGBA', img_back.size)
    img_draw = ImageDraw.Draw(img_poly)
    img_draw.polygon(polygon, fill, outline)
    img_back.paste(img_poly, mask=img_poly)
    return img_back

def get_relative_polygon(polygon, origin: Point, downsample=1):
    """
    Translate the polygon to relative to a point.

    Parameters
    ----------
    polygon : Sequence[Point]
        Polygon points.

    origin : Point
        The new origin the polygons points shall be relative to.

    downsample : int, optional
        Layer downsample >= 1 (Default: 1)

    Returns
    -------
    tuple(Point)
        New polygon with points relative to origin.
    """
    rel_polygon = []
    for point in polygon:
        rel_polygon.append(Point((point.x - origin.x) / downsample,
                                 (point.y - origin.y) / downsample))

    return tuple(rel_polygon)

## 4d — data/camelyon/datamodel.py 

In [12]:
import os
import csv
from collections import defaultdict, OrderedDict, namedtuple
from typing import Tuple, Sequence, Any

from PIL import Image
import openslide
import xml.etree.ElementTree as Xml

# from data.camelyon.cam_utils import Point, get_relative_polygon, draw_polygon, find_files

_RawAnnotation = namedtuple('RawAnnotation', 'name type_ part_of_group color polygon')

class Annotation:
    """Annotation class to provide access to a tumor annotation.

    Annotations can be displayed as an image with the annotation polygon put over the
    annotated section.


    Attributes
    ----------
    slide : Slide
        Slide the annotation belongs to.

    name : str
        Name of the annotation.

    type_ : str
        The type of the annotation specified in the annotation file.

    part_of_group: str
        The group of the annotation specified in the annotation file.

    color : tuple of int or str
        Annotation color as specified in the annotation file.

    polygon : sequence of Point
        A sequence of points annotating the tumor area.
    """

    def __init__(self, slide: 'Slide', name: str, type_: str, part_of_group: str,
                 color: Any, polygon: Sequence[Point]):
        """

        Parameters
        ----------
        slide : Slide
            Slide the annotation belongs to.

        name : str
            Name of the annotation.

        type_ : str
            The type of the annotation specified in the annotation file.

        part_of_group: str
            The group of the annotation specified in the annotation file.

        color : tuple of int or str
            Annotation color as specified in the annotation file.

        polygon : Sequence of Point
            A sequence of points annotating the tumor area.


        See Also
        --------
        PIL.ImageColor
        """
        self.slide = slide
        self.name = name
        self.type = type_
        self.part_of_group = part_of_group
        self.color = color
        self.polygon = polygon

    def __repr__(self):
        return '{}({!r}, {!r}, {!r}, {!r}, {!r}, {!r})'.format(
            type(self).__name__,
            self.slide,
            self.name,
            self.type,
            self.part_of_group,
            self.color,
            self.polygon
        )

    def __str__(self):
        return '{}(slide={!r}, name={!r}, polygon size={!r})'.format(
            type(self).__name__,
            self.slide.name,
            self.name,
            len(self.polygon)
        )

    def get_boundaries(self, level, padding=0):
        """
        Return the annotation boundaries.

        Parameters
        ----------
        level : int
            Layer

        padding : int, optional
            Add additional pixels to the boundaries of the Annotation. (Default: 0)


        Returns
        -------
        origin : (int, int)
            Coordinates of the top left corner of the annotation on the specified layer.

        size : (int, int)
            Annotation width and height on the specified layer.

        """
        x = int(min([p.x for p in self.polygon]) - padding)
        y = int(min([p.y for p in self.polygon]) - padding)
        width = int(max([p.x for p in self.polygon]) - x + padding)
        height = int(max([p.y for p in self.polygon]) - y + padding)

        downsample = self.slide.level_downsamples[level]

        origin = Point(x, y)
        size = (int(width / downsample), int(height / downsample))

        return origin, size

    def get_image(self, *, level=4, padding=100, fill=(50, 50, 50, 80)) -> Image.Image:
        """
        Create an image of the annotated tissue section overlayed with the annotation polygon.

        The polygon's outline `color` will be set to the color attribute of the
        `Annotation` itself. The `fill` color can be specified via the parameter `fill`.

        Parameters
        ----------
        level : int, optional
            Slide level/layer used to create the image.

        padding : int, optional
            Padding added to either side of the image in pixel. Padding is added on layer
            0 and will be downsacled if a `level` higher than 0 is passed.

        fill : tuple of int or str, optional
            Annotation color used to fill the polygon.
            (Default: (50, 50, 50, 80), a dark gray).

        Returns
        -------
        Image.Image
            Image picturing the annotated section from the slide with annotation overlay.

        See Also
        --------
        PIL.ImageColor
        """
        origin, image_size = self.get_boundaries(level, padding)
        downsample = self.slide.level_downsamples[level]

        return draw_polygon(self.slide.read_region(origin, level, image_size),
                            get_relative_polygon(self.polygon, origin,
                                                 downsample),
                            fill=fill,
                            outline=self.color)

def _get_raw_annotations(filename):
    """
    Read all annotation data from an ASAP XML file.

    Parameters
    ----------
    filename : str
        File name of the annotation XML-File.

    Returns
    -------
    Tuple[_RawAnnotation]
        Parsed annotation form XML-File.
    """
    #logger.debug('Reading annotation data from {}', filename)
    tree = Xml.parse(filename)
    root = tree.getroot()
    annotations = []

    for annotation in root.iter('Annotation'):
        # all annotation points sorted by the `Order` attribute
        polygon = (Point(float(c.attrib['X']), float(c.attrib['Y'])) for c in
                   sorted(annotation.iter('Coordinate'),
                          key=lambda x: int(x.attrib['Order'])))

        annotations.append(_RawAnnotation(
            annotation.attrib['Name'].replace(' ', ''),
            annotation.attrib['Type'],
            annotation.attrib['PartOfGroup'],
            annotation.attrib['Color'],
            tuple(polygon)
        ))

    return tuple(annotations)

class Slide(openslide.OpenSlide):
    """
    Wrapper class for openslide.OpenSlide.

    In addition to the OpenSlide itself this class holds information like name and
    possible annotations and stage of the slide itself.

    Attributes
    ----------
    name : str
        Name of the slide.

    stage : str or None
        pN-stage of the slide (None for CAMELYON16 slides).

    has_tumor : bool
        True if the slide has annotations or a non negative pN-stage.

    is_annotated : bool
        True if the slide has annotation.

    See Also
    --------
    openslide.OpenSlide
    """

    def __init__(self, name, filename, annotation_filename=None, stage=None,
                 otsu_thresholds=None):
        """
        Parameters
        ----------
        name : str
            Slide name. Usually the filename without extension.

        filename : str
            Relative or absolute path to slide file.

        annotation_filename : str or None, optional
            Relative or absolute path to an annotation XML file. (Default: None)

        stage : str or None, optional
            nP-stage for CAMELYON17 slides. Leave `None` for CAMELYON16 slides.
            (Default: None)

        otsu_thresholds : dict of float or None, optional
            Dictionary with otsu thresholds for each level. (Default: None)
            Dictionary does not have to be exhaustive e.g.: {0: 6.333, 5: 7.0}
        """
        super().__init__(filename)
        self.name = name
        self._filename = filename
        self._annotation_filename = annotation_filename
        self.stage = stage
        self.is_annotated = self._annotation_filename is not None
        self.has_tumor = self.is_annotated or (
            self.stage is not None and self.stage != 'negative')
        self._otsu_thresholds = otsu_thresholds if otsu_thresholds is not None else {}
        self._annotations = None

    @property
    def annotations(self) -> Tuple[Annotation]:
        """
        Return a tuple of all annotations.

        Returns
        -------
        tuple of Annotation
            All annotations belonging to this instance of `Slide` as a tuple.
        """
        if self._annotations is None:
            if self.is_annotated:
                raw_annotations = _get_raw_annotations(self._annotation_filename)
                self._annotations = tuple(Annotation(self, *x) for x in raw_annotations)
            else:
                self._annotations = ()

        return self._annotations

    def get_full_slide(self, level) -> Image.Image:
        """
        Return the full image of a slide layer.

        Returns
        -------
        Image.Image
            Complete slide on layer `level`.
        """
        return self.read_region((0, 0), level, self.level_dimensions[level])

    def get_otsu_threshold(self, level):
        """
        Return pre-calculated otsu threshold of a layer.

        Parameters
        ----------
        level : int
            Slide layer

        Returns
        -------
        otsu_threshold: float or None
            Otsu threshold of layer `level` or None if not pre-calculated.
        """
        if level in self._otsu_thresholds:
            return self._otsu_thresholds[level]
        else:
            return None

    def __repr__(self):
        if self.is_annotated:
            repr_str = "{}({!r}, {!r}, {!r}, {!r})"
        else:
            repr_str = "{}({!r}, {!r}, {!r})"

        return repr_str.format(type(self).__name__,
                               self.name,
                               self._filename,
                               self.stage,
                               self._annotation_filename)

class SlideManager:
    """
    Provide access to slices from CAMELYON16.

    Attributes
    ----------
    negative_slides : tuple of Slide
        All slides that do not have annotations.

    annotated_slides : tuple of Slide
        All slides that have annotations.
    """

    def __init__(self, *, data_dir, otsu_fname):
        """
        Initialize the CAMELYON data set.

        Parameters
        ----------
        data_dir : str
            Path to the CAMELYON16 directory.
        """

        self._slides = OrderedDict()
        self.slide_paths = OrderedDict()
        self.annotation_paths = OrderedDict()
        self.stages = OrderedDict()
        self.negative_slides = tuple()
        self.annotated_slides = tuple()
        self.test_slides = tuple()

        self.num_positive_train = 0
        self.num_negative_train = 0

        data_dir = os.path.expanduser(data_dir)
        self._path = {
            'dir': data_dir,
            'negative': os.path.join(data_dir, 'training/normal'),
            'positive': os.path.join(data_dir, 'training/tumor'),
            'annotations': os.path.join(data_dir, 'training/lesion_annotations'),
            'test': os.path.join(data_dir, 'testing/images'),
            'test_annotations': os.path.join(data_dir, 'testing/lesion_annotations'),
            'otsu': os.path.join(data_dir, otsu_fname)
        }
        self.__load_data()

    def __load_data(self):
        """Load slides."""

        # Negative slides
        self.otsu_thresholds = defaultdict(dict)
        try:
            with open(self._path['otsu'], 'r') as f:
                reader = csv.DictReader(f)
                for line in reader:
                    self.otsu_thresholds[line['name']][int(line['level'])] = float(
                        line['threshold'])
        except FileNotFoundError:
            print('No pre-calculated otsu thresholds found.')

        slide_files = find_files('*.tif', self._path['negative'])
        for file_name, slide_path in sorted(slide_files.items()):
            slide_name, _, _ = file_name.partition('.')
            slide = Slide(slide_name, slide_path,
                          otsu_thresholds=self.otsu_thresholds[slide_name])

            if slide_name in self._slides:
                raise RuntimeError(f'Slide "{slide_name}" already exists! ({slide_path})')

            self._slides[slide_name] = slide
            self.slide_paths[slide_name] = slide_path
            self.negative_slides += (slide,)
            self.num_negative_train += 1

        # Positive (tumor) slides
        slide_files = find_files('*.tif', self._path['positive'])
        for file_name, slide_path in sorted(slide_files.items()):
            slide_name, _, _ = file_name.partition('.')
            annotation_path = os.path.join(self._path['annotations'],
                                           f'{slide_name}.xml')
            if not os.path.exists(annotation_path):
                raise FileNotFoundError(annotation_path)
            slide = Slide(slide_name, slide_path, otsu_thresholds=self.otsu_thresholds[slide_name],
                annotation_filename=annotation_path)

            if slide_name in self._slides:
                raise RuntimeError(f'Slide "{slide_name}" already exists! ({slide_path})')

            self._slides[slide_name] = slide
            self.slide_paths[slide_name] = slide_path
            self.annotation_paths[slide_name] = annotation_path
            self.annotated_slides += (slide,)
    
            self.num_positive_train += 1

        # test slides
        slide_files = find_files('*.tif', self._path['test'])
        for file_name, slide_path in sorted(slide_files.items()):
            slide_name, _, _ = file_name.partition('.')
            annotation_path = os.path.join(self._path['test_annotations'],
                                           f'{slide_name}.xml')
            if not os.path.exists(annotation_path):
                slide = Slide(slide_name, slide_path,
                          otsu_thresholds=self.otsu_thresholds[slide_name])
            else:
                slide = Slide(slide_name, slide_path, otsu_thresholds=self.otsu_thresholds[slide_name], 
                    annotation_filename=annotation_path)
                
                self.annotation_paths[slide_name] = annotation_path

            if slide_name in self._slides:
                raise RuntimeError(f'Slide "{slide_name}" already exists! ({slide_path})')

            self._slides[slide_name] = slide
            self.slide_paths[slide_name] = slide_path
            self.test_slides += (slide,)


    @property
    def slides(self) -> Tuple[Slide]:
        """
        Return all slides as tuple.

        Returns
        -------
        tuple of Slide
            All slides managed by the instance of `SlideManager`.
        """
        return tuple(self._slides.values())

    @property
    def slide_names(self) -> Tuple[str]:
        """
        Return slide names as tuple.

        Returns
        -------
        tuple of str
            Slide names of all slides managed by the instance of `SlideManager`.
        """
        return tuple(self._slides.keys())

    def get_slide_names_subset(self, train=True) -> Tuple[str]:
        """
        Return slide names as tuple.

        Returns
        -------
        tuple of str
            Slide names of all slides managed by the instance of `SlideManager`.
        """
        if train:
            names = tuple(name for name in self._slides.keys() if 'test' not in name)
        else:
            names = tuple(name for name in self._slides.keys() if 'test' in name)

        return names

    def get_slide(self, name) -> Slide:
        """
        Retrieve a slide by its name.

        Parameters
        ----------
        name : str
            Slide name.


        Returns
        -------
        Slide
            Slide-Object with the name passed.
        """
        return self._slides[name]

    def __repr__(self):
        return '{}(cam16_dir={!r}, cam17_dir={!r})'.format(type(self).__name__,
                                                           self._path['cam16']['dir'],
                                                           self._path['cam17']['dir'])

    def __str__(self):
        return 'SlideManager contains: {} Slides ({} annotated; {} negative)'.format(
            len(self.slides),
            len(self.annotated_slides),
            len(self.negative_slides))

## 4e — data/camelyon/cam_methods.py 

In [13]:
import math
import numpy as np
import datetime

from skimage.draw import polygon as ski_polygon
from skimage.measure import label as ski_label

# from data.camelyon.datamodel import Slide
# from data.camelyon.cam_utils import ProgressBar

def remove_alpha_channel(image: np.ndarray) -> np.ndarray:
    """
    Remove the alpha channel of an image.

    Parameters
    ----------
    image : np.ndarray
        RGBA image as numpy array with W×H×C dimensions.

    Returns
    -------
    np.ndarray
        RGB image as numpy array
    """
    if len(image.shape) == 3 and image.shape[2] == 4:
        return image[::, ::, 0:3:]
    else:
        return image

def rgb2gray(rgb: np.ndarray) -> np.ndarray:
    """
    Convert RGB color image to a custom gray scale for HE-stained WSI

    Parameters
    ----------
    rgb : np.ndarray
        Color image.

    Returns
    -------
    np.ndarray
        Gray scale image as float64 array.
    """
    gray = 1.0 * rgb[::, ::, 0] + rgb[::, ::, 2] - (
        (1.0 * rgb[::, ::, 0] + rgb[::, ::, 1] + rgb[::, ::, 2])
        / 1.5)
    gray[gray < 0] = 0
    gray[gray > 255] = 255
    return gray

def create_otsu_mask_by_threshold(image: np.ndarray, threshold) -> np.ndarray:
    """
    Create a binary mask separating fore and background based on the otsu threshold.

    Parameters
    ----------
    image : np.ndarray
        Gray scale image as array W×H dimensions.

    threshold : float
        Upper Otsu threshold value.

    Returns
    -------
    np.ndarray
        The generated binary masks has value 1 in foreground areas and 0s everywhere
        else (background)
    """
    otsu_mask = image > threshold
    otsu_mask2 = image > threshold * 0.25

    otsu_mask2_labeled = ski_label(otsu_mask2)
    for i in range(1, otsu_mask2_labeled.max()):
        if otsu_mask[otsu_mask2_labeled == i].sum() == 0:
            otsu_mask2_labeled[otsu_mask2_labeled == i] = 0
    otsu_mask3 = otsu_mask2_labeled
    otsu_mask3[otsu_mask3 > 0] = 1

    return otsu_mask3.astype(np.uint8)

def _otsu_by_hist(hist, bin_centers) -> float:
    """
    Return threshold value based on Otsu's method using an images histogram.

    Based on skimage's threshold_otsu method without histogram generation.

    Parameters
    ----------
    hist : np.ndarray
        Histogram of a gray scale input image.

    bin_centers: np.ndarray
        Centers of the histogram's bins.

    Returns
    -------
    threshold : float
        Upper threshold value. All pixels with an intensity higher than
        this value are assumed to be foreground.

    References
    ----------
    Wikipedia, http://en.wikipedia.org/wiki/Otsu's_Method

    See Also
    --------
    skimage.filters.threshold_otsu
    """
    hist = hist.astype(float)

    # class probabilities for all possible thresholds
    weight1 = np.cumsum(hist)
    weight2 = np.cumsum(hist[::-1])[::-1]

    # class means for all possible thresholds
    mean1 = np.cumsum(hist * bin_centers) / weight1
    mean2 = (np.cumsum((hist * bin_centers)[::-1]) / weight2[::-1])[::-1]

    # Clip ends to align class 1 and class 2 variables:
    # The last value of `weight1`/`mean1` should pair with zero values in
    # `weight2`/`mean2`, which do not exist.
    variance12 = weight1[:-1] * weight2[1:] * (mean1[:-1] - mean2[1:]) ** 2

    idx = np.argmax(variance12)
    threshold = bin_centers[:-1][idx]
    return threshold

def add_dict(left, right):
    """
    Merge two dictionaries by adding common items.

    Parameters
    ----------
    left: dict
        Left dictionary.

    right
        Right dictionary

    Returns
    -------
    dict
        Resulting dictionary
    """
    return {k: left.get(k, 0) + right.get(k, 0) for k in left.keys() | right.keys()}

def get_otsu_threshold(slide: Slide, level=0, step_size=1000) -> float:
    """
    Calculate the otsu threshold by reading in the slide in chunks.

    To avoid memory overflows the slide image will be loaded in by chunks of the size
    $slide width × `step_size`$. A histogram will be generated of these chunks that will
    be used to calculate the otsu threshold based on skimage's `threshold_otsu` function.

    Parameters
    ----------
    slide : Slide
        Whole slide image slide

    level : int
        Level/layer of the `slide` to be used. Use of level ≠ 0 is not advised, see notes.

    step_size : int
        Each chunk loaded will have the size $slide-width × `step_size`$ on the level 0
        slide. For higher levels the step will be downsampled accordingly (e.g.: with a
        `step_size` of 1000 and `level` of 1 and a downsample factor of 2 the actual size
        of each chunk is $level-1-slide width × 500$.

    Returns
    -------
    otsu_threshold : float
        Upper threshold value. All pixels with an intensity higher than
        this value are assumed to be foreground.
    """

    size = slide.level_dimensions[0]
    downsample = slide.level_downsamples[level]

    # dictionary with all unique values and counts of the whole slide
    slide_count_dict = {}
    for i, y in enumerate(range(0, size[1], step_size)):

        # check if next step exceeds the image height and adjust it if needed
        cur_step = step_size if size[1] - y > step_size else size[1] - y

        # read in the image and transform to gray scale
        start, cut_size = (0, y), (int(size[0] / downsample), int(cur_step / downsample))
        a_img_cut = np.asarray(slide.read_region(start, level, cut_size))
        a_img_cut = rgb2gray(a_img_cut)

        # get unique values and their count
        chunk_count_dict = dict(zip(*np.unique(a_img_cut, return_counts=True)))

        # add those values and count to the dictionary
        slide_count_dict = add_dict(slide_count_dict, chunk_count_dict)

    # transform dictionary back to a arrays and calculate otsu threshold
    unique_values, counts = tuple(np.asarray(x) for x in zip(*slide_count_dict.items()))
    threshold = _otsu_by_hist(counts, unique_values)

    return threshold

def create_tumor_mask(slide: Slide, level, bounds=None):
    """Create a tumor mask for a slide or slide section.

    If `bounds` is given the tumor mask of only the section of the slide will be
    calculated.


    Parameters
    ----------
    slide : Slide
        Tissue slide.

    level : int
        Slide layer.

    bounds : tuple, optional
        Boundaries of a section as: ((x, y), (width, height))
        Where x and y are coordinates of the top left corner of the slide section on
        layer 0 and width and height the dimensions of the section on the specific
        layer `level`.  (Default: None)


    Returns
    -------
    tumor_mask : np.ndarray
        Binary tumor mask of the specified section. Healthy tissue is represented by 0,
        cancerous by 1.
    """
    if bounds is None:
        start_pos = (0, 0)
        size = slide.level_dimensions[level]
    else:
        start_pos, size = bounds

    mask = np.zeros((size[1], size[0]), dtype=np.uint8)
    downsample = slide.level_downsamples[level]

    for i, annotation in enumerate(slide.annotations):
        c_values, r_values = list(zip(*annotation.polygon))
        r = np.array(r_values, dtype=np.float32)
        r -= start_pos[1]
        r /= downsample
        r = np.array(r + 0.5, dtype=np.int32)

        c = np.array(c_values, dtype=np.float32)
        c -= start_pos[0]
        c /= downsample
        c = np.array(c + 0.5, dtype=np.int32)

        rr, cc = ski_polygon(r, c, shape=mask.shape)
        mask[rr, cc] = 1

    return mask

def split_slide(slide: Slide, lvl, otsu_threshold,
                fg_perc_thresh, tile_size, overlap):
    """
    Create tiles from a slide.

    Iterator over the slide in `tile_size`×`tile_size` Tiles. For every tile an otsu mask
    is created and summed up. Only tiles with sums over the percental threshold
    `fg_perc_thresh` will be yield.

    Parameters
    ----------
    slide : Slide
        Input Slide.

    lvl : int
        Layer to produce tiles from.

    otsu_threshold : float
        Otsu threshold of the whole slide on layer `level`.

    fg_perc_thresh : float, optional
        Minimum percentage, 0 to 1, of pixels with tissue per tile. (Default 0.01; 1%)

    tile_size : int
        Pixel size of one side of a square tile the image will be split into.
        (Default: 256)

    overlap : int, optional
        Count of pixel overlapping between two tiles. (Default: 30)

    Yields
    -------
    image_tile : np.ndarray
        Array of shape (`tile_size`, `tile_size`).

    bounds : tuple
        Tile boundaries on layer 0: ((x, y), (width, height))
    """
    if tile_size <= overlap:
        raise ValueError("Overlap has to be smaller than the tile size.")
    if overlap < 0:
        raise ValueError("Overlap can not be negative.")
    if otsu_threshold < 0:
        raise ValueError("Otsu threshold can not be negative.")
    if not 0.0 <= fg_perc_thresh <= 1.0:
        raise ValueError("Foreground threshold has to be between 0 and 1")

    width0, height0 = slide.level_dimensions[0]
    downsample = slide.level_downsamples[lvl]

    # Tile size on level 0
    tile_size0 = int(tile_size * downsample + 0.5)
    overlap0 = int(overlap * downsample + 0.5)

    # Minimum number of foreground pixels to be considered as foreground
    min_fg_count = tile_size ** 2 * fg_perc_thresh

    # We take patches into account if they belong to the foreground or to tumor tissue.
    # Once `num_pos_tiles_threshold` tumor patches have been found, do not verify whether patches belong
    # to tumor tissue anymore to save time.
    num_pos_tiles = 0
    num_pos_tiles_threshold = 100
    
    skip_pos_mask_calc = False
    # Loop through WSI rows and columns
    for y in range(0, height0, tile_size0 - overlap0):

        # Compute number of tumor pixels in row
        if skip_pos_mask_calc:
            n_tumor_pixels_row = 0
        else:
            if slide.has_tumor:
                mask_row = create_tumor_mask(slide, lvl, ((0, y), (width0, tile_size)))
                n_tumor_pixels_row = np.sum(mask_row)
            else:
                n_tumor_pixels_row = 0
            
        for x in range(0, width0, tile_size0 - overlap0):

            # Only proceed if tumor exists in row
            if n_tumor_pixels_row > 0:
                if lvl != 0:
                    mask_this = create_tumor_mask(slide, lvl, ((x, y), (tile_size, tile_size)))
                    pos_count = np.sum(mask_this)
                if lvl == 0:
                    pos_count = np.sum(mask_row[:,x:(x+tile_size)])

                if pos_count > 0:
                    num_pos_tiles +=1
                    if num_pos_tiles > num_pos_tiles_threshold:
                        skip_pos_mask_calc = True

            else:
                pos_count = 0

            tile = np.asarray(slide.read_region((x, y), lvl, (tile_size, tile_size)))
            otsu_mask = create_otsu_mask_by_threshold(rgb2gray(tile), otsu_threshold)

            fg_count = np.sum(otsu_mask)
            if fg_count >= min_fg_count or pos_count > 0:
                yield remove_alpha_channel(tile), ((x, y), (tile_size0, tile_size0))

## data/camelyon/camelyon_dataset.py

In [14]:
import os
import random
import h5py
import numpy as np
import torch
from torch.utils.data import Dataset, Sampler
from torchvision import transforms

# from .datamodel import SlideManager
# from .cam_methods import remove_alpha_channel

class PatchSampler(Sampler):

    FILL_TOKEN = -1
    SLIDE_END_TOKEN = -2

    def __init__(self, bounds, num_samples=None, batch_size=1):
        self.bounds = bounds
        self.num_samples = num_samples
        self.batch_size = batch_size
        self.num_slides = self.bounds.shape[0]
    
    def __len__(self):
        return self.num_samples

    def __iter__(self):

        slide_idx = list(range(self.num_slides))
        self.all_patch_idx = []
        for slide_id in slide_idx:

            row = self.bounds.iloc[slide_id]
            start_id = row['start_id']
            end_id = row['end_id']

            patch_idx = list(range(start_id, end_id+1))
            num_patches = len(patch_idx)

            # Add tokens to fill up batch
            remainder = (num_patches + 1) % self.batch_size # +1 extra patch
            num_to_add = self.batch_size - remainder# if remainder else 0
            patch_idx = patch_idx + [self.FILL_TOKEN] * num_to_add

            # Add token to identify end of slide
            patch_idx.append(self.SLIDE_END_TOKEN)
            self.all_patch_idx.extend(patch_idx)

        return iter(self.all_patch_idx)


class CamelyonImages(Dataset):

    def __init__(self, data_dir, otsu_fname, coords_df, lvl, tile_size):

        self.slide_man = SlideManager(data_dir=data_dir, otsu_fname=otsu_fname)
        self.coords_df = coords_df

        self.lvl = lvl
        self.tile_size = tile_size

        transform_list = [
            transforms.Lambda(lambda x: remove_alpha_channel(np.asarray(x))),
            transforms.ToPILImage(),
            transforms.CenterCrop(224),
            transforms.ToTensor()
        ]
        self.transform = transforms.Compose(transform_list)

        self.current_slide_name = None
        self.current_slide = None
    
    def __len__(self):
        return len(self.coords_df)

    def __getitem__(self, i):

        data = {}
        is_empty = i < 0
        if not is_empty:
            row = self.coords_df.iloc[i]
            slide_name, x, y, pos_id = row[['name', 'x', 'y', 'pos_id']]

            if slide_name != self.current_slide_name:
                slide = self.slide_man.get_slide(slide_name)
                
                self.current_slide_name = slide_name
                self.current_slide = slide
            else:
                slide = self.current_slide

            patch = slide.read_region((x, y), self.lvl, (self.tile_size, self.tile_size))
            data['patch'] = self.transform(patch)
            data['label'] = int(slide.has_tumor)
            data['pos_id'] = pos_id
            data['slide_name'] = slide_name
        else:
            # Set dummy data except for label, which is used to identify dummy
            data['patch'] = torch.empty((3, 224, 224))
            data['label'] = -1
            data['pos_id'] = 9999
            data['slide_name'] = ''
        data['data_id'] = i
        return data


class CamelyonFeatures(Dataset):

    def open_hdf5(self):
        self.dataset = h5py.File(self.data_dir, 'r')

    def select_slides(self):
        h5_data = h5py.File(self.data_dir, 'r')
        self.slide_names = list(h5_data.keys())
        self.data_len = len(self.slide_names)
        h5_data.close()

    def __init__(self, conf, train=True):

        self.tasks = conf.tasks

        filename = conf.train_fname if train else conf.test_fname
        self.data_dir = os.path.join(conf.data_dir, filename)

        self.select_slides()
    
    def __len__(self):
        return self.data_len

    def __getitem__(self, i):
        
        if not hasattr(self, 'dataset'):
            self.open_hdf5()

        slide_name = self.slide_names[i]

        slide = self.dataset[slide_name]
        patches = slide['img'][:]
        label = slide.attrs['label']

        data_dict = {'input': patches}
        for task in self.tasks.values():
            data_dict[task['name']] = label

        return data_dict

## data/camelyon/extract_feat.py

In [15]:
if False:
     #!/usr/bin/env python
    import os
    import h5py
    from pathlib import Path
    import argparse
    import yaml
    import pandas as pd
    import torch
    
    from torch.utils.data import DataLoader
    from pretraining.model.byol_model import BYOLModel
    
    # from data.camelyon.camelyon_dataset import CamelyonImages, PatchSampler
    
    os.environ["CUDA_VISIBLE_DEVICES"] = "0"
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    parser = argparse.ArgumentParser(
        description="Extract SSL features from foreground patches for each slide"
    )
    
    parser.add_argument('--train', dest='is_train', action='store_true')
    parser.add_argument('--test', dest='is_train', action='store_false')
    parser.set_defaults(is_train=True)
    
    parser.add_argument(
        "--lvl",
        type=int,
        default=0,
        help="Choose the magnification level (0: highest) for feature extraction"
    )
    parser.add_argument(
        "--otsu_lvl",
        type=int,
        default=0,
        help="Choose the magnification level for otsu threshold"
    )
    parser.add_argument(
        "--tile_size",
        type=int,
        default=256,
        help="Choose the tile size"
    )
    parser.add_argument(
        "--batch_size",
        type=int,
        default=128,
        help="Choose the batch size"
    )
    parser.add_argument(
        "--num_workers",
        type=int,
        default=8,
        help="Number of workers for data loading"
    )
    parser.add_argument(
        "data_dir",
        help="The directory where the CAMELYON16 dataset is located"
    )
    parser.add_argument(
        "otsu_fname",
        help="The name of the file that holds Otsu thresholds."
    )
    parser.add_argument(
        "bounds_dir",
        help="The directory where the bounds file is located"
    )
    parser.add_argument(
        "coords_dir",
        help="The directory where the coords file is located"
    )
    parser.add_argument(
        "model_dir",
        help="The directory where the model checkpoint is located"
    )
    parser.add_argument(
        "feat_save_dir",
        help="The directory where the features shall be located"
    )
    
    args = parser.parse_args()
    
    # Get arguments
    train = args.is_train
    lvl = args.lvl
    otsu_lvl = args.otsu_lvl
    tile_size = args.tile_size
    batch_size = args.batch_size
    num_workers = args.num_workers
    data_dir = args.data_dir
    otsu_fname = args.otsu_fname
    bounds_dir = args.bounds_dir
    coords_dir = args.coords_dir
    model_dir = args.model_dir
    feat_save_dir = args.feat_save_dir
    
    # Define dataset
    bounds_df = pd.read_pickle(bounds_dir)
    coords_df = pd.read_pickle(coords_dir)
    sampler = PatchSampler(bounds_df, batch_size=batch_size)
    dataset = CamelyonImages(data_dir, otsu_fname, coords_df, lvl, tile_size)
    dataloader = DataLoader(dataset, batch_size=batch_size, sampler=sampler, num_workers=num_workers)
    
    h5file = h5py.File(feat_save_dir, "w")
    
    # Load pre-trained model
    with open(Path('pretraining/config/train_config.yaml'), 'r') as f:
        config = yaml.safe_load(f)
    net = BYOLModel(config)
    checkpoint = torch.load(model_dir, map_location=device)
    loaded_dict = checkpoint['model']
    prefix = 'module.'
    n_clip = len(prefix)
    adapted_dict = {k[n_clip:]: v for k, v in loaded_dict.items()
                if k.startswith(prefix)}
    net.load_state_dict(adapted_dict, strict=True)
    net = net.online_network.encoder
    net.to(device)
    
    net.eval()
    with torch.no_grad():
        current_slide = None
        feature_list = []
        pos_idx_list = []
        num_processed = 0
        for data in dataloader:
            # Get data
            patches = data['patch'].to(device)
            pos_idx = data['pos_id'].to(device)
            data_idx = data['data_id'].to(device)
            slide_names = [name for name in data['slide_name'] if name]
            if len(slide_names) > 0:
                slide_label = data['label'].max() # either 0 or 1, but not negative
                slide_name = slide_names[0] # slide names are same after filtering
    
            # Check if new slide
            is_new_slide = slide_name != current_slide
            if is_new_slide:
                feature_list = []
                pos_idx_list = []
                current_slide = slide_name
    
            # Extract patches from batch (removes empty elements)
            stopper_idx = torch.where(data_idx < 0, 1, 0).nonzero()
            num_neg_idx = stopper_idx.shape[0]
            if num_neg_idx > 0:
                stop_id = stopper_idx[0]
                patches = patches[:stop_id]
                pos_idx = pos_idx[:stop_id]
    
            # Extract features
            if patches.shape[0] > 0:
                features = net(patches)
                b, n_feat = features.shape[:2]
                features = features.view(b, n_feat)
    
                feature_list.append(features)
                pos_idx_list.append(pos_idx)
    
            is_last_patch = data_idx[-1] == PatchSampler.SLIDE_END_TOKEN
            if is_last_patch:
                num_processed += 1
                print("Nr. slides processed: ", num_processed)
    
                features_np = torch.cat(feature_list, 0).cpu().numpy()
                pos_idx_np = torch.cat(pos_idx_list, 0).cpu().numpy()
                
                # Save as HDF5 file
                slide_grp = h5file.create_group(slide_name)
                slide_grp.create_dataset('img', data=features_np, compression="gzip", compression_opts=9)
                slide_grp.create_dataset('pos', data=pos_idx_np, compression="gzip", compression_opts=9)
                slide_grp.attrs['label'] = slide_label
    
    h5file.close()
    print("Stored features successfully!")


    
    pass

## data/camelyon/ostsu.py

In [16]:
if False:
    import csv
    import argparse
    import multiprocessing as mp
    
    # from datamodel import SlideManager, Slide
    # from data.camelyon.cam_methods import get_otsu_threshold
    
    parser = argparse.ArgumentParser(
        description="Compute Otsu thresholds from WSIs"
    )
    parser.add_argument(
        "--lvl",
        type=int,
        default=0,
        help="Choose the magnification level (0: highest) from which to compute thresholds"
    )
    parser.add_argument(
        "--n_worker",
        type=int,
        default=16,
        help="Number of processes to spawn to parallelize computations"
    )
    parser.add_argument(
        "data_dir",
        help="The directory where the CAMELYON16 dataset is located"
    )
    parser.add_argument(
        "otsu_fname",
        help="The directory where to store otsu thresholds."
    )
    
    args = parser.parse_args()
    
    # Get arguments
    lvl = args.lvl
    n_worker = args.n_worker
    data_dir = args.data_dir
    otsu_fname = args.otsu_fname
    
    # Create slide manager to access all slides
    slide_man = SlideManager(data_dir=data_dir, otsu_fname=otsu_fname)
    
    # Multiprocessing function
    def get_slide_threshold(name):
        """
        Obtains a slide and computes the otsu threshold
        """
        slide_path = slide_man.slide_paths[name]
        slide = Slide(name, slide_path)
    
        threshold = get_otsu_threshold(slide, level=lvl, step_size=1000)
    
        del slide
        return name, lvl, threshold
    
    # Calculating the Otsu threshold can take a long time
    # depending on the magnification level, thus parallelize.
    pool = mp.Pool(n_worker)
    slide_thresholds = list(
        pool.map(get_slide_threshold, slide_man.slide_names)
    )
    
    # Write thresholds to file
    f = open(out_dir, "w")
    
    writer = csv.writer(f)
    header = ['name', 'level', 'threshold']
    writer.writerow(header)
    
    for slide_name, level, threshold in slide_thresholds:
        writer.writerow([slide_name, level, threshold])
    
    f.close()
    print("Done saving thresholds!")
    
    pass

## data/camelyon/foreground.py

In [17]:
if False:
        #!/usr/bin/env python
    import os
    import argparse
    from tqdm import tqdm
    import pandas as pd
    import pickle
    import multiprocessing as mp
    
    # from datamodel import SlideManager
    # from cam_methods import split_slide
    
    parser = argparse.ArgumentParser(
        description="Compute foreground coordinates for each slide"
    )
    
    parser.add_argument('--train', dest='is_train', action='store_true')
    parser.add_argument('--test', dest='is_train', action='store_false')
    parser.set_defaults(is_train=True)
    
    parser.add_argument(
        "--lvl",
        type=int,
        default=0,
        help="Choose the magnification level (0: highest) for foreground computation"
    )
    parser.add_argument(
        "--otsu_lvl",
        type=int,
        default=0,
        help="Choose the magnification level for otsu threshold"
    )
    parser.add_argument(
        "--tile_size",
        type=int,
        default=256,
        help="Choose the tile size."
    )
    parser.add_argument(
        "--fg_perc_thresh",
        type=float,
        default=0.01,
        help="Minimum percentage of foreground pixels so that tile is considered foreground."
    )
    parser.add_argument(
        "--overlap",
        type=int,
        default=0,
        help="Overlap between tiles."
    )
    parser.add_argument(
        "--n_worker",
        type=int,
        default=16,
        help="Number of processes to spawn to parallelize computations"
    )
    parser.add_argument(
        "data_dir",
        help="The directory where the CAMELYON16 dataset is located"
    )
    parser.add_argument(
        "otsu_fname",
        help="The name of the file that holds Otsu thresholds."
    )
    parser.add_argument(
        "out_dir",
        help="Directory where foreground coordinates shall be stored. " + \
        "Filenames will be coords_{train/test}.pkl and bounds_{train/test}.pkl"
    )
    
    args = parser.parse_args()
    
    # Get arguments
    train = args.is_train
    lvl = args.lvl
    otsu_lvl = args.otsu_lvl
    tile_size = args.tile_size
    fg_perc_thresh = args.fg_perc_thresh
    overlap = args.overlap
    n_worker = args.n_worker
    data_dir = args.data_dir
    otsu_fname = args.otsu_fname
    out_dir = args.out_dir
    
    subset = 'train' if train else 'test'
    bounds_path = os.path.join(out_dir, 'bounds_' + subset + '.pkl')
    coords_path = os.path.join(out_dir, 'coords_' + subset + '.pkl')
    
    def get_foreground_coords(name):
        slide = slide_man.get_slide(name)
        otsu_threshold = slide.get_otsu_threshold(otsu_lvl)
        tile_iter = split_slide(slide, lvl, otsu_threshold, fg_perc_thresh, tile_size, overlap)
        
        x_vals, y_vals = [], []
        for _, bounds in tile_iter:
            x, y = bounds[0]
            x_vals.append(x)
            y_vals.append(y)
        names = [name] * len(x_vals)
        print("Finished slide: ", name)
    
        return x_vals, y_vals, names
    
    slide_man = SlideManager(data_dir=data_dir, otsu_fname=otsu_fname)
    slide_names = slide_man.get_slide_names_subset(train=train)
    
    # Computing of foreground coordinates can take a long time, thus parallelize
    pool = mp.Pool(n_worker)
    fg_coords = list(tqdm(
        pool.imap(get_foreground_coords, slide_names), total=len(slide_names)
    ))
    
    # Create lists to be populated
    start_idx, end_idx = [], []
    all_idx, pos_idx = [], []
    x_vals, y_vals = [], []
    names = []
    
    start = 0
    for slide_id, slide_coords in enumerate(fg_coords):
        for patch_id, (x, y, name) in enumerate(zip(*slide_coords)):
            x_vals.append(x)
            y_vals.append(y)
    
            all_idx.append(start + patch_id)
            pos_idx.append(patch_id)
            
            names.append(name)
        
        print("Finished slide #", slide_id)
    
        end = start + patch_id
        
        start_idx.append(start)
        end_idx.append(end)
    
        start = end + 1
    
    # Create dataframes
    lvls = [lvl] * len(start_idx)
    bounds_df = pd.DataFrame(
        {
            'level': lvls,
            'names': slide_names,
            'start_id': start_idx,
            'end_id': end_idx
        }
    )
    coords_df = pd.DataFrame(
        {
            'id': all_idx,
            'pos_id': pos_idx,
            'name': names,
            'x': x_vals,
            'y': y_vals
        }
    )
    
    # Save dataframes
    bounds_file = open(bounds_path, 'wb')
    pickle.dump(bounds_df, bounds_file)
    bounds_file.close()
    
    coords_file = open(coords_path, 'wb')
    pickle.dump(coords_df, coords_file)
    coords_file.close()
    
    print("Done storing foreground coordinates.")
        
    pass

# Luna dataset

## Cellule 1 — Explorer la structure du dataset

In [18]:
if luna_extraction:
    import os
    from pathlib import Path
    import pandas as pd
    
    data_root = Path('/kaggle/input/datasets/vafaeii/luna16')
    
    # 1. Contenu du dossier racine
    print("=== Contenu racine ===")
    for item in sorted(data_root.iterdir()):
        print(f"  {'DIR' if item.is_dir() else 'FILE'} {item.name}")
    
    # 2. Contenu du premier sous-dossier trouvé
    print("\n=== Contenu du premier sous-dossier ===")
    subdirs = [x for x in sorted(data_root.iterdir()) if x.is_dir()]
    if subdirs:
        first_dir = subdirs[0]
        print(f"Dans '{first_dir.name}':")
        items = sorted(first_dir.iterdir())
        for item in items[:10]:
            print(f"  {'DIR' if item.is_dir() else 'FILE'} {item.name}")
        if len(items) > 10:
            print(f"  ... et {len(items)-10} autres fichiers")
    
    # 3. Chercher tous les .mhd récursivement
    print("\n=== Fichiers .mhd ===")
    mhd_files = list(data_root.rglob('*.mhd'))
    print(f"Total .mhd trouvés : {len(mhd_files)}")
    if mhd_files:
        print("Exemples (5 premiers) :")
        for f in mhd_files[:5]:
            print(f"  {f.relative_to(data_root)}")
        sample = mhd_files[0]
        print(f"\nPour '{sample.name}':")
        print(f"  parent       = {sample.parent.name}")
        print(f"  parent.parent= {sample.parent.parent.name}")
    
    # 4. Chercher les CSV
    print("\n=== Fichiers CSV ===")
    csv_files = list(data_root.rglob('*.csv'))
    for csv in csv_files:
        print(f"\n--- {csv.relative_to(data_root)} ---")
        df = pd.read_csv(csv)
        print(f"  Shape   : {df.shape}")
        print(f"  Colonnes: {df.columns.tolist()}")
        print(f"  Head(2) :\n{df.head(2).to_string()}")
    pass

##  Cellule 2 — Statistiques détaillées avant d'extraire

In [19]:
if luna_extraction:
    import pandas as pd
    import numpy as np
    from pathlib import Path
    
    data_root = Path('/kaggle/input/datasets/vafaeii/luna16')
    
    candidates  = pd.read_csv(data_root / 'candidates.csv')
    annotations = pd.read_csv(data_root / 'annotations.csv')
    
    # Construire le mapping uid → subset
    mhd_files = list(data_root.rglob('*.mhd'))
    series_to_subset = {f.stem: f.parent.name for f in mhd_files}
    print(f"Total scans .mhd : {len(series_to_subset)}")
    
    # Vérifier combien de candidats ont un scan correspondant
    valid_uids  = set(series_to_subset.keys())
    cand_uids   = set(candidates['seriesuid'].unique())
    matched     = valid_uids & cand_uids
    missing     = cand_uids - valid_uids
    print(f"UIDs dans candidates.csv     : {len(cand_uids)}")
    print(f"UIDs avec scan .mhd trouvé  : {len(matched)}")
    print(f"UIDs sans scan (manquants)  : {len(missing)}")
    
    # Distribution train/test
    train_subsets = {f'subset{i}' for i in range(6)}
    test_subsets  = {f'subset{i}' for i in range(6, 10)}
    
    train_uids = [u for u in matched if series_to_subset[u] in train_subsets]
    test_uids  = [u for u in matched if series_to_subset[u] in test_subsets]
    print(f"\nScans train (subset0-5) : {len(train_uids)}")
    print(f"Scans test  (subset6-9) : {len(test_uids)}")
    
    # Distribution des candidats par classe
    cands_matched = candidates[candidates['seriesuid'].isin(matched)]
    print(f"\nCandidats totaux (scans présents) : {len(cands_matched)}")
    print(f"  class=0 (faux positifs) : {(cands_matched['class']==0).sum()}")
    print(f"  class=1 (vrais nodules) : {(cands_matched['class']==1).sum()}")
    
    # Candidats par scan — stats
    per_scan = cands_matched.groupby('seriesuid').size()
    print(f"\nCandidats par scan :")
    print(f"  min={per_scan.min()}, max={per_scan.max()}, "
          f"mean={per_scan.mean():.0f}, median={per_scan.median():.0f}")
    
    # Labels scan-level (a au moins 1 nodule confirmé dans annotations.csv)
    annot_uids = set(annotations['seriesuid'].unique())
    train_pos  = sum(1 for u in train_uids if u in annot_uids)
    train_neg  = len(train_uids) - train_pos
    test_pos   = sum(1 for u in test_uids  if u in annot_uids)
    test_neg   = len(test_uids)  - test_pos
    print(f"\nLabel scan-level (présence nodule confirmé) :")
    print(f"  Train : {train_pos} positifs / {train_neg} négatifs")
    print(f"  Test  : {test_pos} positifs / {test_neg} négatifs")
    
    # Taille d'un scan pour estimer le temps d'extraction
    import os
    sample_mhd = mhd_files[0]
    sample_raw = sample_mhd.with_suffix('.raw')
    size_mb = sample_mhd.stat().st_size / 1e6
    if sample_raw.exists():
        size_mb += sample_raw.stat().st_size / 1e6
    print(f"\nTaille d'un scan (mhd+raw) : ~{size_mb:.1f} MB")
    print(f"Estimation RAM pour 1 scan  : ~{size_mb:.0f} MB")
    print(f"Temps extraction estimé     : "
          f"~{len(matched) * 2 / 60:.0f} min (à 2s/scan sur GPU)")
    pass

## Cellule 3 — Vérifier un scan avant l'extraction

In [20]:
if luna_extraction:
    import SimpleITK as sitk
    import numpy as np
    import pandas as pd
    from pathlib import Path
    
    data_root  = Path('/kaggle/input/datasets/vafaeii/luna16')
    candidates = pd.read_csv(data_root / 'candidates.csv')
    
    # Prendre le premier scan
    mhd_files       = list(data_root.rglob('*.mhd'))
    sample_mhd      = mhd_files[0]
    uid             = sample_mhd.stem
    
    print(f"UID : {uid}")
    print(f"Path: {sample_mhd.relative_to(data_root)}")
    
    # Charger le scan
    itk_img   = sitk.ReadImage(str(sample_mhd))
    img_array = sitk.GetArrayFromImage(itk_img)  # (D, H, W)
    
    print(f"\n--- Infos scan ---")
    print(f"Size ITK (W,H,D) : {itk_img.GetSize()}")
    print(f"Array shape (D,H,W): {img_array.shape}")
    print(f"Spacing (mm)     : {itk_img.GetSpacing()}")
    print(f"Origin  (mm)     : {itk_img.GetOrigin()}")
    print(f"dtype            : {img_array.dtype}")
    print(f"HU range         : [{img_array.min()}, {img_array.max()}]")
    
    # Candidats pour ce scan
    scan_cands = candidates[candidates['seriesuid'] == uid]
    print(f"\n--- Candidats pour ce scan ---")
    print(f"Total       : {len(scan_cands)}")
    print(f"class=1     : {(scan_cands['class']==1).sum()}")
    print(f"class=0     : {(scan_cands['class']==0).sum()}")
    
    # Tester la conversion de coordonnées sur le premier candidat
    row = scan_cands.iloc[0]
    world_coord = (float(row['coordX']), float(row['coordY']), float(row['coordZ']))
    try:
        vc = itk_img.TransformPhysicalPointToContinuousIndex(world_coord)
        x_vox, y_vox, z_vox = int(round(vc[0])), int(round(vc[1])), int(round(vc[2]))
        D, H, W = img_array.shape
        print(f"\n--- Test conversion coordonnées ---")
        print(f"World (mm) : {world_coord}")
        print(f"Voxel      : ({x_vox}, {y_vox}, {z_vox})")
        print(f"Dans bounds: x={0<=x_vox<W}, y={0<=y_vox<H}, z={0<=z_vox<D}")
    
        # Extraire un patch 50x50 autour de ce candidat
        PATCH_SIZE = 50
        half = PATCH_SIZE // 2
        patch = img_array[z_vox,
                          max(0, y_vox-half):min(H, y_vox+half),
                          max(0, x_vox-half):min(W, x_vox+half)]
        print(f"\n--- Patch extrait ---")
        print(f"Shape avant padding : {patch.shape}")
        # Padding si bord
        ph = PATCH_SIZE - patch.shape[0]
        pw = PATCH_SIZE - patch.shape[1]
        if ph > 0 or pw > 0:
            patch = np.pad(patch,
                           ((ph//2, ph-ph//2), (pw//2, pw-pw//2)),
                           mode='constant', constant_values=img_array.min())
        print(f"Shape après padding : {patch.shape}")
        print(f"HU range dans patch : [{patch.min()}, {patch.max()}]")
    except Exception as e:
        print(f"Erreur conversion : {e}")
    
    # Estimer la taille du HDF5 final
    N_CANDIDATES_KEPT  = 200   # après subsampling (on décidera dans la prochaine cellule)
    N_SCANS            = 888
    FEAT_DIM           = 2048
    size_mb = N_SCANS * N_CANDIDATES_KEPT * FEAT_DIM * 4 / 1e6
    print(f"\n--- Estimation taille HDF5 ---")
    print(f"Avec {N_CANDIDATES_KEPT} candidats/scan × {N_SCANS} scans × {FEAT_DIM} dims")
    print(f"Taille estimée (avant compression) : {size_mb:.0f} MB")
    print(f"Taille estimée (après gzip)        : ~{size_mb*0.3:.0f} MB")
    pass

## Cellule 4 — Extraction des features et sauvegarde HDF5

In [21]:
if luna_extraction:
    # ============================================================
    # Extraction des features LUNA16 → HDF5
    # Run once — ~30 min sur GPU Kaggle
    # ============================================================
    import os, h5py, gc
    import numpy as np
    import torch
    import SimpleITK as sitk
    import pandas as pd
    from pathlib import Path
    from torchvision.models import resnet50, ResNet50_Weights
    import torch.nn as nn
    
    os.environ["CUDA_VISIBLE_DEVICES"] = "0"
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Device : {device}")
    
    # --- Paramètres ---
    DATA_ROOT       = Path('/kaggle/input/datasets/vafaeii/luna16')
    OUTPUT_HDF5     = '/kaggle/working/luna16_features.hdf5'
    PATCH_SIZE      = 50
    BATCH_SIZE      = 512       # patches envoyés au GPU en une fois
    MAX_NEG_PER_SCAN = 9999  # valeur arbitrairement grande = pas de limite
    HU_MIN, HU_MAX  = -1000, 400  # clipping HU standard poumons
    
    # --- Encoder ResNet50 ---
    # Conv1 adapté à 1 canal (CT grayscale)
    encoder = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
    encoder.conv1 = nn.Conv2d(1, 64, kernel_size=7,
                               stride=2, padding=3, bias=False)
    encoder.fc = nn.Identity()   # output = 2048-d
    encoder = encoder.to(device).eval()
    print("Encoder chargé")
    
    # --- Données ---
    candidates  = pd.read_csv(DATA_ROOT / 'candidates.csv')
    annotations = pd.read_csv(DATA_ROOT / 'annotations.csv')
    annot_uids  = set(annotations['seriesuid'].unique())
    
    VALID_SUBSETS = {f'subset{i}' for i in range(10)}
    mhd_files = [f for f in DATA_ROOT.rglob('*.mhd')
             if f.parent.name in VALID_SUBSETS]
    series_to_path   = {f.stem: f for f in mhd_files}
    series_to_subset = {f.stem: f.parent.name for f in mhd_files}
    
    valid_uids = sorted(set(series_to_path.keys()) &
                        set(candidates['seriesuid'].unique()))
    print(f"Scans à traiter : {len(valid_uids)}")
    
    # --- Fonctions utilitaires ---
    def clip_and_normalize(patch_int16):
        """Clip HU, normalise en [0,1], retourne float32."""
        p = patch_int16.astype(np.float32)
        p = np.clip(p, HU_MIN, HU_MAX)
        p = (p - HU_MIN) / (HU_MAX - HU_MIN)
        return p
    
    def extract_patch(img_array, itk_img, x_mm, y_mm, z_mm):
        """Retourne patch numpy [PATCH_SIZE, PATCH_SIZE] ou None si hors bornes."""
        try:
            vc = itk_img.TransformPhysicalPointToContinuousIndex(
                     (float(x_mm), float(y_mm), float(z_mm)))
        except Exception:
            return None
        xv, yv, zv = int(round(vc[0])), int(round(vc[1])), int(round(vc[2]))
        D, H, W = img_array.shape
        if not (0 <= zv < D):
            return None
        half = PATCH_SIZE // 2
        y0, y1 = max(0, yv-half), min(H, yv+half)
        x0, x1 = max(0, xv-half), min(W, xv+half)
        patch = img_array[zv, y0:y1, x0:x1]
        # Padding si le patch touche un bord
        ph = PATCH_SIZE - patch.shape[0]
        pw = PATCH_SIZE - patch.shape[1]
        if ph > 0 or pw > 0:
            fill = int(HU_MIN)  # air = -1000 HU
            patch = np.pad(patch,
                           ((ph//2, ph-ph//2), (pw//2, pw-pw//2)),
                           mode='constant', constant_values=fill)
        return patch


    import os
    if os.path.exists(OUTPUT_HDF5):
        os.remove(OUTPUT_HDF5)
        print(f"Ancien HDF5 supprimé")
        
    # --- Extraction ---
    h5file = h5py.File(OUTPUT_HDF5, 'w')
    rng    = np.random.default_rng(seed=0)
    
    n_done, n_skipped = 0, 0
    
    for scan_idx, uid in enumerate(valid_uids):
    
        scan_cands = candidates[candidates['seriesuid'] == uid]
        pos_cands  = scan_cands[scan_cands['class'] == 1]
        neg_cands  = scan_cands[scan_cands['class'] == 0]
    
        # Subsample négatifs
        if len(neg_cands) > MAX_NEG_PER_SCAN:
            neg_cands = neg_cands.sample(
                n=MAX_NEG_PER_SCAN, random_state=int(rng.integers(1e6)))
    
        selected = pd.concat([pos_cands, neg_cands]).reset_index(drop=True)
    
        # Charger le scan CT
        itk_img   = sitk.ReadImage(str(series_to_path[uid]))
        img_array = sitk.GetArrayFromImage(itk_img)  # (D, H, W), int16
    
        # Extraire les patches
        patches, classes = [], []
        for _, row in selected.iterrows():
            patch = extract_patch(img_array,
                                  itk_img,
                                  row['coordX'],
                                  row['coordY'],
                                  row['coordZ'])
            if patch is None:
                continue
            patches.append(clip_and_normalize(patch))
            classes.append(int(row['class']))
    
        # Libérer le scan immédiatement
        del itk_img, img_array
        gc.collect()
    
        if len(patches) == 0:
            n_skipped += 1
            continue
    
        # Encoder les patches par batch sur GPU
        patches_np = np.stack(patches)               # [N, 50, 50]
        patches_t  = torch.from_numpy(patches_np
                         ).unsqueeze(1).float()       # [N, 1, 50, 50]
    
        all_feats = []
        with torch.no_grad():
            for i in range(0, len(patches_t), BATCH_SIZE):
                batch = patches_t[i:i+BATCH_SIZE].to(device)
                feats = encoder(batch)                # [B, 2048]
                all_feats.append(feats.cpu().numpy())
    
        features_np = np.concatenate(all_feats, axis=0)  # [N, 2048]
        classes_np  = np.array(classes, dtype=np.int32)   # [N]
        label = 1 if int(selected['class'].max()) == 1 else 0
    
        # Écrire dans HDF5
        grp = h5file.create_group(uid)
        grp.create_dataset('img', data=features_np,
                           compression='gzip', compression_opts=6)
        grp.create_dataset('pos', data=classes_np,
                           compression='gzip', compression_opts=6)
        grp.attrs['label']  = label
        grp.attrs['subset'] = series_to_subset[uid]
    
        n_done += 1
        if (scan_idx + 1) % 50 == 0 or scan_idx == 0:
            n_patches = features_np.shape[0]
            print(f"[{scan_idx+1:4d}/{len(valid_uids)}] {uid[:40]}... "
                  f"| {n_patches} patches | label={label} "
                  f"| subset={series_to_subset[uid]}")
    
        # Nettoyer
        del patches_t, all_feats, features_np
        gc.collect()
    
    h5file.close()
    print(f"\nTerminé — {n_done} scans extraits, {n_skipped} ignorés")
    print(f"HDF5 sauvegardé : {OUTPUT_HDF5}")
    print(f"Taille fichier  : {os.path.getsize(OUTPUT_HDF5)/1e6:.1f} MB")
    pass

## Cellule 7 — Vérifier le HDF5 en détail avant l'entraînement

In [22]:
if luna_extraction:
    import h5py
    import numpy as np
    
    h5 = h5py.File('/kaggle/working/luna16_features.hdf5', 'r')
    
    TRAIN_SUBSETS = {f'subset{i}' for i in range(6)}
    TEST_SUBSETS  = {f'subset{i}' for i in range(6, 10)}
    
    train_uids = [uid for uid in h5.keys()
                  if h5[uid].attrs.get('subset','') in TRAIN_SUBSETS]
    test_uids  = [uid for uid in h5.keys()
                  if h5[uid].attrs.get('subset','') not in TRAIN_SUBSETS]
    
    print(f"Scans train (subset0-5) : {len(train_uids)}")
    print(f"Scans test  (subset6-9) : {len(test_uids)}")
    
    # Labels
    train_labels = [h5[uid].attrs['label'] for uid in train_uids]
    test_labels  = [h5[uid].attrs['label'] for uid in test_uids]
    print(f"\nTrain — positifs : {sum(train_labels)}, "
          f"négatifs : {len(train_labels)-sum(train_labels)}")
    print(f"Test  — positifs : {sum(test_labels)}, "
          f"négatifs : {len(test_labels)-sum(test_labels)}")
    
    # Vérifier shape features sur quelques scans
    print(f"\nExemples de shapes :")
    for uid in list(h5.keys())[:3]:
        img   = h5[uid]['img'][:]
        pos   = h5[uid]['pos'][:]
        label = h5[uid].attrs['label']
        subset= h5[uid].attrs['subset']
        print(f"  {uid[:45]}...")
        print(f"    img shape : {img.shape}  "
              f"pos shape : {pos.shape}  "
              f"label={label}  subset={subset}")
        print(f"    pos unique values : {np.unique(pos)}")
        print(f"    features range    : [{img.min():.3f}, {img.max():.3f}]")
    
    h5.close()
    pass

## Bloc A — Dataset class (calqué exactement sur CamelyonFeatures)

In [23]:
# data/luna16/luna_dataset.py
import os, h5py
import numpy as np
import torch
from torch.utils.data import Dataset

class LUNAFeatures(Dataset):

    TRAIN_SUBSETS = {f'subset{i}' for i in range(6)}
    TEST_SUBSETS  = {f'subset{i}' for i in range(6, 10)}

    def open_hdf5(self):
        self.dataset = h5py.File(self.data_path, 'r')

    def select_scans(self):
        h5 = h5py.File(self.data_path, 'r')
        subsets = self.TRAIN_SUBSETS if self.train else self.TEST_SUBSETS
        self.scan_names = []
        for uid in h5.keys():
            if h5[uid].attrs.get('subset', '') not in subsets:
                continue
            label   = int(h5[uid].attrs['label'])
            pos_arr = h5[uid]['pos'][:]
            n_pos   = int(pos_arr.sum())
            # Exclure scans label=1 sans candidat positif — signal contradictoire
            if label == 1 and n_pos == 0:
                continue
            self.scan_names.append(uid)
        self.data_len = len(self.scan_names)
        h5.close()
        print(f"{'Train' if self.train else 'Test'} : {self.data_len} scans "
              f"(contradictoires exclus)")

    def __init__(self, conf, train=True):
        self.tasks     = conf.tasks
        self.data_path = conf.data_dir
        self.train     = train
        self.rng       = np.random.default_rng(seed=0)
        self.select_scans()

    def __len__(self):
        return self.data_len

    def __getitem__(self, i):
        if not hasattr(self, 'dataset'):
            self.open_hdf5()

        uid     = self.scan_names[i]
        scan    = self.dataset[uid]
        patches = scan['img'][:]   # [N, 2048]
        pos_arr = scan['pos'][:]   # [N]
        label   = int(scan.attrs['label'])

        if self.train:
            # Garder tous les positifs + 300 négatifs aléatoires
            # → diversité entre epochs, N ~303 > M=200, IPS actif
            pos_idx = np.where(pos_arr == 1)[0]
            neg_idx = np.where(pos_arr == 0)[0]
            if len(neg_idx) > 300:
                neg_idx = self.rng.choice(neg_idx, size=300, replace=False)
            keep = np.concatenate([pos_idx, neg_idx])
            self.rng.shuffle(keep)
            patches = patches[keep]

        data_dict = {'input': torch.from_numpy(patches.astype(np.float32))}
        for task in self.tasks.values():
            data_dict[task['name']] = label
        return data_dict

## Cellule 10c — Ouvrir sans locking

In [24]:
if luna_extraction:
    os.environ['HDF5_USE_FILE_LOCKING'] = 'FALSE'
    import h5py
    import numpy as np
    
    h5_path = '/kaggle/working/luna16_features.hdf5'
    
    # Désactiver le file locking HDF5 — nécessaire quand DataLoader workers 
    # ont encore des handles ouverts en arrière-plan
    import os
    os.environ['HDF5_USE_FILE_LOCKING'] = 'FALSE'
    
    h5 = h5py.File(h5_path, 'r+')
    
    TRAIN_SUBSETS = {f'subset{i}' for i in range(6)}
    TEST_SUBSETS  = {f'subset{i}' for i in range(6, 10)}
    
    n_changed = 0
    for uid in h5.keys():
        pos_array = h5[uid]['pos'][:]
        new_label = 1 if pos_array.max() >= 1 else 0
        old_label = int(h5[uid].attrs['label'])
        if new_label != old_label:
            h5[uid].attrs['label'] = new_label
            n_changed += 1
    
    h5.close()
    print(f"Labels mis à jour : {n_changed} scans modifiés")
    
    # Vérifier la nouvelle distribution
    h5 = h5py.File(h5_path, 'r')
    labels       = [h5[uid].attrs['label'] for uid in h5.keys()]
    train_labels = [h5[uid].attrs['label'] for uid in h5.keys()
                    if h5[uid].attrs.get('subset','') in TRAIN_SUBSETS]
    test_labels  = [h5[uid].attrs['label'] for uid in h5.keys()
                    if h5[uid].attrs.get('subset','') in TEST_SUBSETS]
    
    print(f"\nNouvelle distribution :")
    print(f"Total  — positifs : {sum(labels)}, "
          f"négatifs : {len(labels)-sum(labels)}  "
          f"({100*sum(labels)/len(labels):.1f}% pos)")
    print(f"Train  — positifs : {sum(train_labels)}, "
          f"négatifs : {len(train_labels)-sum(train_labels)}  "
          f"({100*sum(train_labels)/len(train_labels):.1f}% pos)")
    print(f"Test   — positifs : {sum(test_labels)}, "
          f"négatifs : {len(test_labels)-sum(test_labels)}  "
          f"({100*sum(test_labels)/len(test_labels):.1f}% pos)")
    h5.close()
    pass

# Node21

## Cellule NODE21-1 — Explorer la structure

In [25]:
import os
from pathlib import Path
import pandas as pd

data_root = Path('/kaggle/input/datasets/pshikk/node-21-dataset-untampered')

# 1. Contenu racine
print("=== Contenu racine ===")
for item in sorted(data_root.iterdir()):
    n_items = len(list(item.iterdir())) if item.is_dir() else ''
    print(f"  {'DIR' if item.is_dir() else 'FILE'} {item.name}"
          + (f"  ({n_items} items)" if n_items != '' else ''))

# 2. Premier sous-dossier
print("\n=== Premier sous-dossier ===")
subdirs = [x for x in sorted(data_root.iterdir()) if x.is_dir()]
if subdirs:
    first = subdirs[0]
    print(f"Dans '{first.name}' :")
    items = sorted(first.iterdir())
    for item in items[:10]:
        print(f"  {'DIR' if item.is_dir() else 'FILE'} {item.name}")
    if len(items) > 10:
        print(f"  ... et {len(items)-10} autres")

# 3. Tous les formats de fichiers présents
print("\n=== Extensions présentes ===")
from collections import Counter
ext_counts = Counter(
    f.suffix.lower() for f in data_root.rglob('*') if f.is_file()
)
for ext, count in sorted(ext_counts.items(), key=lambda x: -x[1]):
    print(f"  {ext:15s} : {count}")

# 4. CSV / annotations
print("\n=== Fichiers CSV ===")
for csv in sorted(data_root.rglob('*.csv')):
    df = pd.read_csv(csv)
    print(f"\n--- {csv.relative_to(data_root)} ---")
    print(f"  Shape    : {df.shape}")
    print(f"  Colonnes : {df.columns.tolist()}")
    print(df.head(3).to_string())

# 5. Si images — taille d'un sample
print("\n=== Sample image ===")
img_files = list(data_root.rglob('*.png')) + \
            list(data_root.rglob('*.jpg')) + \
            list(data_root.rglob('*.mhd')) + \
            list(data_root.rglob('*.dcm'))
print(f"Total images trouvées : {len(img_files)}")
if img_files:
    sample = img_files[0]
    print(f"Exemple : {sample.relative_to(data_root)}")
    size_mb = sample.stat().st_size / 1e6
    print(f"Taille  : {size_mb:.2f} MB")
    # Essayer d'ouvrir
    try:
        from PIL import Image
        img = Image.open(sample)
        print(f"Mode    : {img.mode}")
        print(f"Size    : {img.size}")
    except Exception:
        pass
    try:
        import SimpleITK as sitk
        itk = sitk.ReadImage(str(sample))
        print(f"ITK size    : {itk.GetSize()}")
        print(f"ITK spacing : {itk.GetSpacing()}")
    except Exception:
        pass

=== Contenu racine ===
  DIR ct_patches  (2 items)
  DIR cxr_images  (2 items)

=== Premier sous-dossier ===
Dans 'ct_patches' :
  DIR nodule_patches
  DIR segmentation

=== Extensions présentes ===
  .mha            : 12136
  .csv            : 4

=== Fichiers CSV ===

--- cxr_images/original_data/filenames_orig_and_new.csv ---
  Shape    : (1134, 3)
  Colonnes : ['original_image_name', 'orig_dataset', 'node21_img_id']
  original_image_name orig_dataset node21_img_id
0        00010496_001  chestxray14         n0239
1        00006281_000  chestxray14         n0342
2        00001404_000  chestxray14         n0996

--- cxr_images/original_data/metadata.csv ---
  Shape    : (5224, 7)
  Colonnes : ['Unnamed: 0', 'height', 'img_name', 'label', 'width', 'x', 'y']
   Unnamed: 0  height   img_name  label  width    x    y
0           0      94  n0239.mha      1     92  776  538
1           1      40  n0342.mha      1     27  215  641
2           2     108  n0996.mha      1    155  695  342

--- 

## Cellule NODE21-2 — Exploration détaillée

In [26]:
import SimpleITK as sitk
import numpy as np
from pathlib import Path
import pandas as pd

data_root = Path('/kaggle/input/datasets/pshikk/node-21-dataset-untampered')

# === 1. CT patches ===
print("=== ct_patches/nodule_patches ===")
nodule_dir = data_root / 'ct_patches' / 'nodule_patches'
nodule_files = list(nodule_dir.rglob('*.mha'))
print(f"Fichiers .mha : {len(nodule_files)}")
if nodule_files:
    sample = nodule_files[0]
    itk    = sitk.ReadImage(str(sample))
    arr    = sitk.GetArrayFromImage(itk)
    print(f"Exemple      : {sample.name}")
    print(f"Shape        : {arr.shape}  (Z, Y, X ou Y, X)")
    print(f"Spacing      : {itk.GetSpacing()}")
    print(f"dtype        : {arr.dtype}")
    print(f"HU range     : [{arr.min()}, {arr.max()}]")
    print(f"Taille fichier: {sample.stat().st_size/1e3:.1f} KB")

print("\n=== ct_patches/segmentation ===")
seg_dir   = data_root / 'ct_patches' / 'segmentation'
seg_files = list(seg_dir.rglob('*.mha'))
print(f"Fichiers .mha : {len(seg_files)}")
if seg_files:
    s   = sitk.ReadImage(str(seg_files[0]))
    arr = sitk.GetArrayFromImage(s)
    print(f"Shape    : {arr.shape}")
    print(f"Unique   : {np.unique(arr)}")

# === 2. CXR images ===
print("\n=== cxr_images ===")
for subdir in ['original_data', 'proccessed_data']:
    cxr_dir   = data_root / 'cxr_images' / subdir
    mha_files = list(cxr_dir.rglob('*.mha'))
    print(f"\n{subdir} : {len(mha_files)} fichiers .mha")
    if mha_files:
        # Trier pour avoir un vrai exemple (pas simulé)
        real = [f for f in mha_files if f.name.startswith('n')]
        sim  = [f for f in mha_files if f.name.startswith('c')]
        print(f"  réels (n*.mha)    : {len(real)}")
        print(f"  simulés (c*.mha)  : {len(sim)}")
        if real:
            s   = sitk.ReadImage(str(real[0]))
            arr = sitk.GetArrayFromImage(s)
            print(f"  Shape réel   : {arr.shape}")
            print(f"  dtype        : {arr.dtype}")
            print(f"  range        : [{arr.min()}, {arr.max()}]")
            print(f"  Taille       : {real[0].stat().st_size/1e6:.2f} MB")
        if sim:
            s   = sitk.ReadImage(str(sim[0]))
            arr = sitk.GetArrayFromImage(s)
            print(f"  Shape simulé : {arr.shape}")
            print(f"  Taille       : {sim[0].stat().st_size/1e6:.2f} MB")

# === 3. Distribution labels dans metadata ===
print("\n=== Distribution labels ===")
meta = pd.read_csv(data_root / 'cxr_images' / 'proccessed_data' / 'metadata.csv')
print(f"label=1 (nodule)   : {(meta['label']==1).sum()}")
print(f"label=0 (no nodule): {(meta['label']==0).sum() if 'label' in meta.columns else 'N/A'}")
print(f"Images uniques     : {meta['img_name'].nunique()}")
print(f"Annotations/image  : {len(meta)/meta['img_name'].nunique():.1f} en moyenne")

# Taille des nodules annotés
print(f"\nTaille nodules (pixels) :")
print(f"  width  — min={meta['width'].min()}, max={meta['width'].max()}, "
      f"mean={meta['width'].mean():.0f}")
print(f"  height — min={meta['height'].min()}, max={meta['height'].max()}, "
      f"mean={meta['height'].mean():.0f}")

=== ct_patches/nodule_patches ===
Fichiers .mha : 1186
Exemple      : 1.3.6.1.4.1.14519.5.2.1.6279.6001.213854687290736562463866711534_dcm_0.mha
Shape        : (50, 50, 50)  (Z, Y, X ou Y, X)
Spacing      : (1.0, 1.0, 1.0)
dtype        : int16
HU range     : [-1023, 773]
Taille fichier: 158.9 KB

=== ct_patches/segmentation ===
Fichiers .mha : 1186
Shape    : (50, 50, 50)
Unique   : [0 1]

=== cxr_images ===

original_data : 4882 fichiers .mha
  réels (n*.mha)    : 1134
  simulés (c*.mha)  : 3748
  Shape réel   : (1024, 1024)
  dtype        : uint8
  range        : [0, 232]
  Taille       : 0.62 MB
  Shape simulé : (3608, 3880)
  Taille       : 20.97 MB

proccessed_data : 4882 fichiers .mha
  réels (n*.mha)    : 1134
  simulés (c*.mha)  : 3748
  Shape réel   : (1024, 1024)
  dtype        : uint16
  range        : [0, 4095]
  Taille       : 1.48 MB
  Shape simulé : (1024, 1024)
  Taille       : 1.28 MB

=== Distribution labels ===
label=1 (nodule)   : 1476
label=0 (no nodule): 3748
Imag

## Cellule NODE21-4 — Trouver où sont les fichiers .mha

In [27]:
import SimpleITK as sitk
import numpy as np
import pandas as pd
from pathlib import Path

data_root = Path('/kaggle/input/datasets/pshikk/node-21-dataset-untampered')

# Trouver TOUS les .mha récursivement et leur emplacement exact
all_mha = list(data_root.rglob('*.mha'))
print(f"Total .mha trouvés : {len(all_mha)}")

# Grouper par dossier parent
from collections import Counter
parent_counts = Counter(f.parent.relative_to(data_root).as_posix() 
                        for f in all_mha)
print("\nRépartition par dossier :")
for folder, count in sorted(parent_counts.items(), key=lambda x: -x[1]):
    print(f"  {folder:60s} : {count}")

# Exemples dans chaque dossier
print("\nExemples de noms :")
seen_parents = set()
for f in all_mha:
    parent = f.parent.relative_to(data_root).as_posix()
    if parent not in seen_parents:
        seen_parents.add(parent)
        print(f"  [{parent}]")
        siblings = list(f.parent.glob('*.mha'))[:5]
        for s in siblings:
            print(f"    {s.name}")

# Lire un sample de chaque type
print("\n=== Lecture samples ===")
for folder, count in sorted(parent_counts.items(), key=lambda x: -x[1])[:4]:
    folder_path = data_root / folder
    files = list(folder_path.glob('*.mha'))
    if files:
        try:
            itk = sitk.ReadImage(str(files[0]))
            arr = sitk.GetArrayFromImage(itk)
            print(f"\n[{folder}]")
            print(f"  Exemple  : {files[0].name}")
            print(f"  Shape    : {arr.shape}")
            print(f"  dtype    : {arr.dtype}")
            print(f"  range    : [{arr.min()}, {arr.max()}]")
            print(f"  Taille   : {files[0].stat().st_size/1e6:.2f} MB")
        except Exception as e:
            print(f"  Erreur   : {e}")

Total .mha trouvés : 12136

Répartition par dossier :
  cxr_images/original_data/images                              : 4882
  cxr_images/proccessed_data/images                            : 4882
  ct_patches/segmentation                                      : 1186
  ct_patches/nodule_patches                                    : 1186

Exemples de noms :
  [ct_patches/segmentation]
    1.3.6.1.4.1.14519.5.2.1.6279.6001.176030616406569931557298712518_seg_3.mha
    1.3.6.1.4.1.14519.5.2.1.6279.6001.362762275895885013176610377950_seg_0.mha
    1.3.6.1.4.1.14519.5.2.1.6279.6001.229096941293122177107846044795_seg_4.mha
    1.3.6.1.4.1.14519.5.2.1.6279.6001.328789598898469177563438457842_seg_6.mha
    1.3.6.1.4.1.14519.5.2.1.6279.6001.107351566259572521472765997306_seg_0.mha
  [ct_patches/nodule_patches]
    1.3.6.1.4.1.14519.5.2.1.6279.6001.213854687290736562463866711534_dcm_0.mha
    1.3.6.1.4.1.14519.5.2.1.6279.6001.286217539434358186648717203667_dcm_2.mha
    1.3.6.1.4.1.14519.5.2.1.6279.60

## Cellule NODE21-5 — Vérifier la correspondance original/processed et créer le plan négatifs

In [28]:
import SimpleITK as sitk
import numpy as np
import pandas as pd
from pathlib import Path

data_root  = Path('/kaggle/input/datasets/pshikk/node-21-dataset-untampered')
proc_dir   = data_root / 'cxr_images' / 'proccessed_data' / 'images'
orig_dir   = data_root / 'cxr_images' / 'original_data'   / 'images'

meta_real = pd.read_csv(data_root / 'cxr_images' / 'proccessed_data' / 'metadata.csv')
meta_sim  = pd.read_csv(data_root / 'cxr_images' / 'proccessed_data' / 'simulated_metadata.csv')

proc_files = sorted(proc_dir.glob('*.mha'))
orig_files = sorted(orig_dir.glob('*.mha'))

# 1. Noms disponibles
proc_names = {f.name for f in proc_files}
orig_names = {f.name for f in orig_files}
n_names    = {f.name for f in proc_files if f.name.startswith('n')}
c_names    = {f.name for f in proc_files if f.name.startswith('c')}
print(f"proc/images — n*.mha : {len(n_names)}, c*.mha : {len(c_names)}")
print(f"orig/images — n*.mha : {len([f for f in orig_names if f.startswith('n')])}, "
      f"c*.mha : {len([f for f in orig_names if f.startswith('c')])}")

# 2. Vérifier que orig et proc ont les mêmes noms
same = proc_names == orig_names
print(f"\nMêmes fichiers orig/proc : {same}")
if not same:
    only_proc = proc_names - orig_names
    only_orig = orig_names - proc_names
    print(f"  Seulement dans proc : {len(only_proc)}")
    print(f"  Seulement dans orig : {len(only_orig)}")

# 3. Idée négatifs : orig_data/c*.mha = radiographie SANS nodule simulé
#    proc_data/c*.mha = même radiographie AVEC nodule simulé
#    → orig_data/c*.mha sont nos négatifs
sample_c = [f for f in orig_files if f.name.startswith('c')][0]
arr_orig  = sitk.GetArrayFromImage(sitk.ReadImage(str(sample_c)))
arr_proc  = sitk.GetArrayFromImage(
    sitk.ReadImage(str(proc_dir / sample_c.name)))

print(f"\n=== Comparaison orig vs proc pour {sample_c.name} ===")
print(f"orig shape : {arr_orig.shape}, range : [{arr_orig.min()}, {arr_orig.max()}]")
print(f"proc shape : {arr_proc.shape}, range : [{arr_proc.min()}, {arr_proc.max()}]")
diff = arr_orig.astype(np.float32) - arr_proc.astype(np.float32)
print(f"diff max   : {np.abs(diff).max():.1f}")
print(f"diff mean  : {np.abs(diff).mean():.4f}")
print(f"Pixels différents : {(np.abs(diff) > 0).sum()} "
      f"({100*(np.abs(diff)>0).mean():.2f}%)")

# 4. Plan dataset final
n_positifs = len(n_names) + len(c_names)   # toutes proc = nodule présent
n_negatifs = len([f for f in orig_names if f.startswith('c')])  # orig c* = sans nodule
print(f"\n=== Plan dataset final ===")
print(f"Positifs (proc/n*.mha + proc/c*.mha) : {n_positifs}")
print(f"Négatifs (orig/c*.mha sans nodule)   : {n_negatifs}")
print(f"Total                                 : {n_positifs + n_negatifs}")
print(f"Ratio pos/neg                         : {n_positifs/n_negatifs:.1f}:1")

# 5. Split train/test — pas de split officiel, on en crée un
# Idée : utiliser les noms pour séparer
all_img_names = sorted(proc_names)
import hashlib
def name_to_split(name, test_ratio=0.2, seed=42):
    h = int(hashlib.md5((name+str(seed)).encode()).hexdigest(), 16)
    return 'test' if (h % 100) < (test_ratio * 100) else 'train'

splits = {name: name_to_split(name) for name in all_img_names}
n_train = sum(1 for v in splits.values() if v == 'train')
n_test  = sum(1 for v in splits.values() if v == 'test')
print(f"\nSplit 80/20 (hash-based) :")
print(f"  Train : {n_train}")
print(f"  Test  : {n_test}")

proc/images — n*.mha : 1134, c*.mha : 3748
orig/images — n*.mha : 1134, c*.mha : 3748

Mêmes fichiers orig/proc : True

=== Comparaison orig vs proc pour c0001.mha ===
orig shape : (1024, 1024), range : [0, 244]
proc shape : (1024, 1024), range : [0, 4094]
diff max   : 4001.0
diff mean  : 2107.1750
Pixels différents : 1048186 (99.96%)

=== Plan dataset final ===
Positifs (proc/n*.mha + proc/c*.mha) : 4882
Négatifs (orig/c*.mha sans nodule)   : 3748
Total                                 : 8630
Ratio pos/neg                         : 1.3:1

Split 80/20 (hash-based) :
  Train : 3902
  Test  : 980


## Cellule NODE21-6 corrigée — Vérifier la vraie structure

In [29]:
import SimpleITK as sitk
import numpy as np
import pandas as pd
from pathlib import Path

data_root = Path('/kaggle/input/datasets/pshikk/node-21-dataset-untampered')
orig_dir  = data_root / 'cxr_images' / 'original_data'  / 'images'
proc_dir  = data_root / 'cxr_images' / 'proccessed_data' / 'images'

meta_real = pd.read_csv(
    data_root / 'cxr_images' / 'proccessed_data' / 'metadata.csv')
meta_sim  = pd.read_csv(
    data_root / 'cxr_images' / 'proccessed_data' / 'simulated_metadata.csv')

# 1. Récapitulatif des fichiers
orig_n = sorted(orig_dir.glob('n*.mha'))   # positifs originaux
orig_c = sorted(orig_dir.glob('c*.mha'))   # négatifs (sans nodule)
proc_n = sorted(proc_dir.glob('n*.mha'))   # positifs preprocessés
proc_c = sorted(proc_dir.glob('c*.mha'))   # négatifs preprocessés

print("=== Fichiers disponibles ===")
print(f"original_data  — n*.mha (positifs) : {len(orig_n)}")
print(f"original_data  — c*.mha (négatifs) : {len(orig_c)}")
print(f"proccessed_data— n*.mha (positifs) : {len(proc_n)}")
print(f"proccessed_data— c*.mha (négatifs) : {len(proc_c)}")

# 2. Vérifier qu'un c*.mha original est vraiment sans nodule
#    → comparer orig/c*.mha vs proc/c*.mha
sample_c_orig = orig_c[0]
sample_c_proc = proc_dir / sample_c_orig.name

arr_orig = sitk.GetArrayFromImage(sitk.ReadImage(str(sample_c_orig)))
arr_proc = sitk.GetArrayFromImage(sitk.ReadImage(str(sample_c_proc)))

print(f"\n=== Comparaison pour {sample_c_orig.name} ===")
print(f"orig shape/range : {arr_orig.shape}, [{arr_orig.min()}, {arr_orig.max()}]")
print(f"proc shape/range : {arr_proc.shape}, [{arr_proc.min()}, {arr_proc.max()}]")

# Si simulated_metadata contient ce fichier → il a un nodule simulé dans proc
in_sim = sample_c_orig.name in meta_sim['img_name'].values
print(f"Dans simulated_metadata : {in_sim}  "
      f"← True = nodule simulé dans proc, False = négatif pur")

# 3. Vérifier combien de c*.mha ont un nodule simulé dans proc
sim_names = set(meta_sim['img_name'].unique())
c_names   = {f.name for f in proc_c}
c_with_nodule    = c_names & sim_names      # c* avec nodule simulé dans proc
c_without_nodule = c_names - sim_names      # c* sans nodule dans proc

print(f"\n=== c*.mha dans proccessed_data ===")
print(f"Total c*.mha                    : {len(c_names)}")
print(f"Avec nodule simulé (proc)       : {len(c_with_nodule)}")
print(f"Sans nodule simulé (vrais nég.) : {len(c_without_nodule)}")

# 4. Plan dataset final
print(f"\n=== Plan dataset final ===")
print(f"Positifs — proc/n*.mha          : {len(proc_n)}")
print(f"  nodules annotés               : {meta_real['img_name'].nunique()}")
print(f"Négatifs — orig/c*.mha          : {len(orig_c)}")
print(f"  (radiographies sans nodule)   ")
print(f"\nTotal                           : {len(proc_n) + len(orig_c)}")
print(f"Ratio pos/neg                   : "
      f"{len(proc_n)/len(orig_c):.2f}:1")

# 5. Split train/test 80/20
import hashlib
all_positifs = [(f.name, 1) for f in proc_n]
all_negatifs = [(f.name, 0) for f in orig_c]
all_samples  = all_positifs + all_negatifs

def to_split(name, seed=42):
    h = int(hashlib.md5((name+str(seed)).encode()).hexdigest(), 16)
    return 'test' if (h % 100) < 20 else 'train'

df = pd.DataFrame(all_samples, columns=['img_name', 'label'])
df['split'] = df['img_name'].apply(to_split)

print(f"\n=== Split 80/20 ===")
for split in ['train', 'test']:
    sub = df[df['split'] == split]
    pos = (sub['label'] == 1).sum()
    neg = (sub['label'] == 0).sum()
    print(f"{split:5s} : {len(sub):4d} images  "
          f"(pos={pos}, neg={neg}, ratio={pos/neg:.2f})")

=== Fichiers disponibles ===
original_data  — n*.mha (positifs) : 1134
original_data  — c*.mha (négatifs) : 3748
proccessed_data— n*.mha (positifs) : 1134
proccessed_data— c*.mha (négatifs) : 3748

=== Comparaison pour c0001.mha ===
orig shape/range : (1024, 1024), [0, 244]
proc shape/range : (1024, 1024), [0, 4094]
Dans simulated_metadata : True  ← True = nodule simulé dans proc, False = négatif pur

=== c*.mha dans proccessed_data ===
Total c*.mha                    : 3748
Avec nodule simulé (proc)       : 3748
Sans nodule simulé (vrais nég.) : 0

=== Plan dataset final ===
Positifs — proc/n*.mha          : 1134
  nodules annotés               : 4882
Négatifs — orig/c*.mha          : 3748
  (radiographies sans nodule)   

Total                           : 4882
Ratio pos/neg                   : 0.30:1

=== Split 80/20 ===
train : 3902 images  (pos=915, neg=2987, ratio=0.31)
test  :  980 images  (pos=219, neg=761, ratio=0.29)


## Cellule NODE21-7 — Dataset class + Config + runner

In [30]:
# data/node21/node21_dataset.py
import os
import numpy as np
import torch
import SimpleITK as sitk
import pandas as pd
import hashlib
from pathlib import Path
from torch.utils.data import Dataset
from torchvision import transforms

class NODE21Dataset(Dataset):
    """
    Classification binaire : nodule présent (1) ou absent (0)
    Positifs : proccessed_data/images/n*.mha  (uint16, preprocessées)
    Négatifs : original_data/images/c*.mha    (uint8, sans nodule simulé)
    Split train/test reproductible par hash sur le nom de fichier.
    """

    CLASSES = ['no_nodule', 'nodule']   # 0, 1
    IMG_SIZE = 1024

    @staticmethod
    def _to_split(name, seed=42):
        h = int(hashlib.md5((name + str(seed)).encode()).hexdigest(), 16)
        return 'test' if (h % 100) < 20 else 'train'

    def __init__(self, conf, train=True):
        self.patch_size   = conf.patch_size
        self.patch_stride = conf.patch_stride
        self.tasks        = conf.tasks

        data_root = Path(conf.data_dir)
        proc_dir  = data_root / 'cxr_images' / 'proccessed_data' / 'images'
        orig_dir  = data_root / 'cxr_images' / 'original_data'   / 'images'

        # Construire la liste (path, label)
        all_samples = (
            [(f, 1) for f in sorted(proc_dir.glob('n*.mha'))] +
            [(f, 0) for f in sorted(orig_dir.glob('c*.mha'))]
        )

        # Filtrer train ou test
        split_name = 'train' if train else 'test'
        self._data = [
            (path, label) for path, label in all_samples
            if self._to_split(path.name) == split_name
        ]

        # Transforms — normalise en [0,1] puis standardise
        # Même pipeline pour uint8 et uint16 grâce à la normalisation manuelle
        if train:
            self.transform = transforms.Compose([
                transforms.RandomHorizontalFlip(),
                transforms.RandomRotation(5),
                transforms.Normalize(mean=[0.5], std=[0.5])
            ])
        else:
            self.transform = transforms.Compose([
                transforms.Normalize(mean=[0.5], std=[0.5])
            ])

        print(f"{'Train' if train else 'Test'} : {len(self._data)} images "
              f"(pos={sum(l for _,l in self._data)}, "
              f"neg={sum(1-l for _,l in self._data)})")

    def __len__(self):
        return len(self._data)

    def __getitem__(self, i):
        path, label = self._data[i]
    
        # Charger
        arr = sitk.GetArrayFromImage(sitk.ReadImage(str(path)))
        arr = arr.astype(np.float32)
    
        # ← AJOUTER CES LIGNES : redimensionner à 1024×1024 si nécessaire
        if arr.shape != (self.IMG_SIZE, self.IMG_SIZE):
            from PIL import Image as PILImage
            pil = PILImage.fromarray(
                ((arr - arr.min()) / (arr.max() - arr.min() + 1e-6) * 255)
                .astype(np.uint8)
            ).resize((self.IMG_SIZE, self.IMG_SIZE), PILImage.BILINEAR)
            arr = np.array(pil, dtype=np.float32) / 255.0
        else:
            # Normaliser en [0,1]
            arr = (arr - arr.min()) / (arr.max() - arr.min() + 1e-6)
    
        # [H, W] → tensor [1, H, W]
        img = torch.from_numpy(arr).unsqueeze(0)
    
        # Normalise
        img = self.transform(img)
    
        # Extraire les patches
        ps, stride = self.patch_size, self.patch_stride
        patches = img.unfold(1, ps[0], stride[0]) \
                     .unfold(2, ps[1], stride[1]) \
                     .permute(1, 2, 0, 3, 4)
        patches = patches.reshape(-1, *patches.shape[2:])
    
        data_dict = {'input': patches}
        for task in self.tasks.values():
            data_dict[task['name']] = label
        return data_dict

# lungCT

## Cellule LungCT-1 — Explorer la structure

In [31]:
import os
from pathlib import Path
import pandas as pd
from collections import Counter

data_root = Path('/kaggle/input/datasets/aubinyoumbi/lungct-diagnosis-tcia-mirrored-untampered')

# 1. Contenu racine
print("=== Contenu racine ===")
for item in sorted(data_root.iterdir()):
    if item.is_dir():
        n = len(list(item.rglob('*')))
        print(f"  DIR  {item.name}  ({n} items récursivement)")
    else:
        print(f"  FILE {item.name}  ({item.stat().st_size/1e6:.2f} MB)")

# 2. Extensions présentes
print("\n=== Extensions présentes ===")
ext_counts = Counter(
    f.suffix.lower() for f in data_root.rglob('*') if f.is_file()
)
for ext, count in sorted(ext_counts.items(), key=lambda x: -x[1]):
    print(f"  {ext:15s} : {count}")

# 3. CSV / metadata
print("\n=== Fichiers CSV / JSON / metadata ===")
for f in sorted(data_root.rglob('*.csv')):
    try:
        df = pd.read_csv(f)
        print(f"\n--- {f.relative_to(data_root)} ---")
        print(f"  Shape    : {df.shape}")
        print(f"  Colonnes : {df.columns.tolist()}")
        print(df.head(3).to_string())
    except Exception as e:
        print(f"  Erreur   : {e}")

for f in sorted(data_root.rglob('*.json'))[:3]:
    print(f"\n--- {f.relative_to(data_root)} ---")
    import json
    with open(f) as fp:
        d = json.load(fp)
    if isinstance(d, dict):
        print(f"  Clés : {list(d.keys())[:10]}")
    elif isinstance(d, list):
        print(f"  Liste de {len(d)} éléments")
        print(f"  Premier : {d[0]}")

# 4. Structure des sous-dossiers (2 niveaux)
print("\n=== Arborescence (2 niveaux) ===")
for lvl1 in sorted(data_root.iterdir()):
    if not lvl1.is_dir():
        continue
    print(f"  {lvl1.name}/")
    lvl2_items = sorted(lvl1.iterdir())
    for lvl2 in lvl2_items[:5]:
        if lvl2.is_dir():
            n = len(list(lvl2.iterdir()))
            print(f"    {lvl2.name}/  ({n} items)")
        else:
            print(f"    {lvl2.name}  ({lvl2.stat().st_size/1e3:.1f} KB)")
    if len(lvl2_items) > 5:
        print(f"    ... et {len(lvl2_items)-5} autres")

# 5. Sample d'image
print("\n=== Sample image ===")
img_extensions = {'.dcm', '.mha', '.mhd', '.nii', '.nii.gz', '.png', '.jpg'}
img_files = [f for f in data_root.rglob('*')
             if f.suffix.lower() in img_extensions]
print(f"Total images : {len(img_files)}")
if img_files:
    sample = img_files[0]
    print(f"Exemple : {sample.relative_to(data_root)}")
    print(f"Taille  : {sample.stat().st_size/1e6:.2f} MB")
    try:
        import SimpleITK as sitk
        import numpy as np
        itk = sitk.ReadImage(str(sample))
        arr = sitk.GetArrayFromImage(itk)
        print(f"Shape   : {arr.shape}")
        print(f"dtype   : {arr.dtype}")
        print(f"range   : [{arr.min()}, {arr.max()}]")
        print(f"Spacing : {itk.GetSpacing()}")
    except Exception:
        pass
    try:
        from PIL import Image
        img = Image.open(sample)
        print(f"PIL mode : {img.mode}, size : {img.size}")
    except Exception:
        pass

=== Contenu racine ===
  DIR  LungCT-Diagnosis  (4870 items récursivement)

=== Extensions présentes ===
  .dcm            : 4682
  .xlsx           : 1
  .docx           : 1
  .csv            : 1
                  : 1

=== Fichiers CSV / JSON / metadata ===

--- LungCT-Diagnosis/metadata.csv ---
  Shape    : (61, 17)
  Colonnes : ['Series UID', 'Collection', '3rd Party Analysis', 'Data Description URI', 'Subject ID', 'Study UID', 'Study Description', 'Study Date', 'Series Description', 'Manufacturer', 'Modality', 'SOP Class Name', 'SOP Class UID', 'Number of Images', 'File Size', 'File Location', 'Download Timestamp']
                                                         Series UID        Collection  3rd Party Analysis                           Data Description URI Subject ID                                                         Study UID                            Study Description  Study Date Series Description        Manufacturer Modality    SOP Class Name              SOP Clas

## Cellule LungCT-2 — Trouver les labels et comprendre la structure complète

In [41]:
import os
import numpy as np
import pandas as pd
import SimpleITK as sitk
from pathlib import Path

data_root = Path('/kaggle/input/datasets/aubinyoumbi/lungct-diagnosis-tcia-mirrored-untampered')

# Trouver les vrais emplacements
print("=== Arborescence complète (3 niveaux) ===")
for lvl1 in sorted(data_root.iterdir()):
    print(f"{lvl1.name}/")
    for lvl2 in sorted(lvl1.iterdir())[:8]:
        if lvl2.is_dir():
            n = len(list(lvl2.iterdir()))
            print(f"  {lvl2.name}/  ({n} items)")
            for lvl3 in sorted(lvl2.iterdir())[:3]:
                if lvl3.is_dir():
                    n3 = len(list(lvl3.iterdir()))
                    print(f"    {lvl3.name}/  ({n3} items)")
                else:
                    print(f"    {lvl3.name}  ({lvl3.stat().st_size/1e3:.1f} KB)")
        else:
            print(f"  {lvl2.name}  ({lvl2.stat().st_size/1e3:.1f} KB)")

# Trouver tous les fichiers metadata/xlsx/csv/docx
print("\n=== Fichiers metadata ===")
for ext in ['*.csv', '*.xlsx', '*.docx', '*.txt', '*.json']:
    for f in sorted(data_root.rglob(ext)):
        print(f"  {f.relative_to(data_root)}  ({f.stat().st_size/1e3:.1f} KB)")

=== Arborescence complète (3 niveaux) ===
LungCT-Diagnosis/
  LungCT-Diagnosis/  (62 items)
    LICENSE  (5.7 KB)
    R_004/  (1 items)
    R_006/  (1 items)
  LungCT-Diagnosis_SurvivalData_journal.pone_.0118261.s010.docx  (31.6 KB)
  Representative-Tumor-Slices.xlsx  (16.3 KB)
  metadata.csv  (29.2 KB)

=== Fichiers metadata ===
  LungCT-Diagnosis/metadata.csv  (29.2 KB)
  LungCT-Diagnosis/Representative-Tumor-Slices.xlsx  (16.3 KB)
  LungCT-Diagnosis/LungCT-Diagnosis_SurvivalData_journal.pone_.0118261.s010.docx  (31.6 KB)


## Cellule LungCT-4 — Lire les labels et la structure

In [42]:
import numpy as np
import pandas as pd
import SimpleITK as sitk
from pathlib import Path

data_root  = Path('/kaggle/input/datasets/aubinyoumbi/lungct-diagnosis-tcia-mirrored-untampered')
lung_root  = data_root / 'LungCT-Diagnosis'   # ← niveau correct
lung_dir   = lung_root / 'LungCT-Diagnosis'   # ← dossiers patients

# 1. metadata.csv
print("=== metadata.csv ===")
meta = pd.read_csv(lung_root / 'metadata.csv')
print(f"Shape    : {meta.shape}")
print(f"Colonnes : {meta.columns.tolist()}")
print(f"Patients : {meta['Subject ID'].nunique()}")
print(meta[['Subject ID', 'Number of Images', 'Study Description']].to_string())

# 2. Excel — labels principaux
print("\n=== Representative-Tumor-Slices.xlsx ===")
try:
    xl = pd.read_excel(lung_root / 'Representative-Tumor-Slices.xlsx',
                       sheet_name=None)
    for sheet_name, df in xl.items():
        print(f"\nSheet '{sheet_name}' : {df.shape}")
        print(f"Colonnes : {df.columns.tolist()}")
        print(df.to_string())
except Exception as e:
    print(f"Erreur xlsx : {e}")
    # Essayer openpyxl direct
    try:
        import openpyxl
        wb = openpyxl.load_workbook(lung_root / 'Representative-Tumor-Slices.xlsx')
        print(f"Sheets : {wb.sheetnames}")
        for sheet in wb.sheetnames:
            ws = wb[sheet]
            print(f"\nSheet '{sheet}' :")
            for row in ws.iter_rows(values_only=True):
                print(row)
    except Exception as e2:
        print(f"Erreur openpyxl : {e2}")

# 3. Structure dossiers patients
print("\n=== Dossiers patients ===")
patient_dirs = sorted([d for d in lung_dir.iterdir()
                       if d.is_dir() and d.name.startswith('R_')])
print(f"Total patients : {len(patient_dirs)}")
print(f"Liste : {[d.name for d in patient_dirs]}")

# 4. Structure d'un patient
p = patient_dirs[0]
print(f"\nStructure '{p.name}' :")
for study in sorted(p.iterdir()):
    print(f"  {study.name}/")
    for series in sorted(study.iterdir()):
        dcms = list(series.rglob('*.dcm'))
        print(f"    {series.name}/  ({len(dcms)} dcm)")

# 5. Lire le volume 3D d'un patient
print(f"\n=== Volume 3D de '{p.name}' ===")
try:
    study  = sorted(p.iterdir())[0]
    series = sorted(study.iterdir())[0]
    reader = sitk.ImageSeriesReader()
    series_ids   = reader.GetGDCMSeriesIDs(str(series))
    dicom_names  = reader.GetGDCMSeriesFileNames(str(series), series_ids[0])
    reader.SetFileNames(dicom_names)
    vol = reader.Execute()
    arr = sitk.GetArrayFromImage(vol)
    print(f"Volume shape : {arr.shape}  (Z, H, W)")
    print(f"HU range     : [{arr.min()}, {arr.max()}]")
    print(f"Spacing      : {vol.GetSpacing()} mm")
    print(f"Slices       : {arr.shape[0]}")
except Exception as e:
    print(f"Erreur volume : {e}")

=== metadata.csv ===
Shape    : (61, 17)
Colonnes : ['Series UID', 'Collection', '3rd Party Analysis', 'Data Description URI', 'Subject ID', 'Study UID', 'Study Description', 'Study Date', 'Series Description', 'Manufacturer', 'Modality', 'SOP Class Name', 'SOP Class UID', 'Number of Images', 'File Size', 'File Location', 'Download Timestamp']
Patients : 61
   Subject ID  Number of Images                            Study Description
0       R_013                67  Diagnostic Pre-Surgery Contrast Enhanced CT
1       R_014                76  Diagnostic Pre-Surgery Contrast Enhanced CT
2       R_004                68  Diagnostic Pre-Surgery Contrast Enhanced CT
3       R_019                63  Diagnostic Pre-Surgery Contrast Enhanced CT
4       R_006               130  Diagnostic Pre-Surgery Contrast Enhanced CT
5       R_022                80  Diagnostic Pre-Surgery Contrast Enhanced CT
6       R_020               150  Diagnostic Pre-Surgery Contrast Enhanced CT
7       R_029           

## Cellule LungCT-5 — Lire le fichier de survie

In [43]:
import pandas as pd
from pathlib import Path

data_root = Path('/kaggle/input/datasets/aubinyoumbi/lungct-diagnosis-tcia-mirrored-untampered')
lung_root = data_root / 'LungCT-Diagnosis'

# Lire le docx avec python-docx
try:
    from docx import Document
    doc = Document(lung_root / 'LungCT-Diagnosis_SurvivalData_journal.pone_.0118261.s010.docx')

    print("=== Texte du document ===")
    for para in doc.paragraphs:
        if para.text.strip():
            print(para.text)

    print("\n=== Tables dans le document ===")
    for t_idx, table in enumerate(doc.tables):
        print(f"\nTable {t_idx} : {len(table.rows)} lignes × {len(table.columns)} colonnes")
        for i, row in enumerate(table.rows):
            cells = [c.text.strip() for c in row.cells]
            print(f"  {cells}")
            if i >= 20:
                print(f"  ... ({len(table.rows)-20} lignes restantes)")
                break

except ImportError:
    print("python-docx non installé — installation...")
    import subprocess
    subprocess.run(['pip', 'install', 'python-docx', '-q'])
    from docx import Document
    doc = Document(lung_root / 'LungCT-Diagnosis_SurvivalData_journal.pone_.0118261.s010.docx')
    for para in doc.paragraphs:
        if para.text.strip():
            print(para.text)
    for t_idx, table in enumerate(doc.tables):
        print(f"\nTable {t_idx} : {len(table.rows)} × {len(table.columns)}")
        for row in table.rows:
            print([c.text.strip() for c in row.cells])
except Exception as e:
    print(f"Erreur : {e}")

python-docx non installé — installation...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 5.3 MB/s eta 0:00:00
Table D. Clinical parameters and feature scores for Cohort 1 patients.

Table 0 : 49 × 11
['Table A. Distribution of study population demographics and imaging parameters by imaging biomarkers in Cohort 2.', 'Table A. Distribution of study population demographics and imaging parameters by imaging biomarkers in Cohort 2.', 'Table A. Distribution of study population demographics and imaging parameters by imaging biomarkers in Cohort 2.', 'Table A. Distribution of study population demographics and imaging parameters by imaging biomarkers in Cohort 2.', 'Table A. Distribution of study population demographics and imaging parameters by imaging biomarkers in Cohort 2.', 'Table A. Distribution of study population demographics and imaging parameters by imaging biomarkers in Cohort 2.', 'Table A. Distribution of study population demographics and imaging parameters by imaging 

## Cellule LungCT-6 — Distribution des labels et plan final

In [45]:
import pandas as pd
import numpy as np
from pathlib import Path
from docx import Document

data_root = Path('/kaggle/input/datasets/aubinyoumbi/lungct-diagnosis-tcia-mirrored-untampered')
lung_root = data_root / 'LungCT-Diagnosis'
lung_dir  = lung_root / 'LungCT-Diagnosis'

# 1. Parser le docx → DataFrame propre
doc    = Document(lung_root / 'LungCT-Diagnosis_SurvivalData_journal.pone_.0118261.s010.docx')
table3 = doc.tables[3]
rows   = [[c.text.strip() for c in row.cells] for row in table3.rows]

df = pd.DataFrame(rows[1:], columns=['patient_id', 'tnm',
                                      'vital_status', 'vital_dup',
                                      'survival_months',
                                      'convexity', 'entropy_ratio'])
df = df[df['patient_id'].str.startswith('Patient_')].copy()
df['patient_id']       = df['patient_id'].str.replace('Patient_', 'R_')
df['survival_months']  = pd.to_numeric(df['survival_months'], errors='coerce')
df['vital_binary']     = df['vital_status'].str.contains('ALIVE').astype(int)
df['convexity']        = pd.to_numeric(df['convexity'], errors='coerce')

# 2. Créer le label TNM simplifié
def tnm_to_label(tnm):
    if pd.isna(tnm) or tnm == 'N/A':
        return None
    if any(tnm.startswith(s) for s in ['1', '2']):
        return 0   # précoce
    if any(tnm.startswith(s) for s in ['3', '4']):
        return 1   # avancé
    return None

df['label'] = df['tnm'].apply(tnm_to_label)

print("=== Labels TNM ===")
print(df[['patient_id', 'tnm', 'label',
          'vital_status', 'survival_months']].to_string())

print(f"\n=== Distribution ===")
print(f"Précoce (I-II)  label=0 : {(df['label']==0).sum()}")
print(f"Avancé  (III-IV) label=1 : {(df['label']==1).sum()}")
print(f"N/A (exclus)             : {df['label'].isna().sum()}")
print(f"Utilisables              : {df['label'].notna().sum()}")

# 3. Mapper vers les dossiers CT
patient_dirs = {d.name for d in lung_dir.iterdir() if d.is_dir()}
df_valid = df[df['label'].notna()].copy()
df_valid['has_ct'] = df_valid['patient_id'].isin(patient_dirs)
print(f"\nPatients avec CT trouvé  : {df_valid['has_ct'].sum()}")
print(f"Patients sans CT         : {(~df_valid['has_ct']).sum()}")
missing = df_valid[~df_valid['has_ct']]['patient_id'].tolist()
if missing:
    print(f"Manquants : {missing}")

# 4. Slices disponibles par patient
print(f"\n=== Slices par patient ===")
meta = pd.read_csv(lung_root / 'metadata.csv')
meta_dict = dict(zip(meta['Subject ID'], meta['Number of Images']))
df_valid['n_slices'] = df_valid['patient_id'].map(meta_dict)
print(f"min={df_valid['n_slices'].min():.0f}, "
      f"max={df_valid['n_slices'].max():.0f}, "
      f"mean={df_valid['n_slices'].mean():.1f}")
print(f"Total slices train (estimé 80%) : "
      f"{int(df_valid['n_slices'].sum() * 0.8)}")

# 5. Excel — slice représentative de la tumeur
xl = pd.read_excel(lung_root / 'Representative-Tumor-Slices.xlsx',
                   header=1)
xl.columns = ['patient_id', 'sop_uid', 'instance_number', 'image_position']
xl = xl[xl['patient_id'].str.startswith('R_', na=False)].copy()
xl['instance_number'] = pd.to_numeric(xl['instance_number'], errors='coerce')
print(f"\n=== Slices tumorales représentatives ===")
print(f"Total : {len(xl)}")
print(xl[['patient_id', 'instance_number', 'image_position']].head(10).to_string())

# Merge
df_final = df_valid.merge(xl[['patient_id', 'instance_number']],
                           on='patient_id', how='left')
print(f"\n=== Dataset final ===")
print(f"Patients avec label + CT + slice tumorale : "
      f"{df_final['instance_number'].notna().sum()}")
print(f"\nPlan extraction :")
print(f"  Stratégie A : toutes les slices du CT (mean={df_valid['n_slices'].mean():.0f}/patient)")
print(f"  Stratégie B : ±N slices autour de la slice tumorale (plus ciblé)")

=== Labels TNM ===
   patient_id  tnm  label vital_status  survival_months
0       R_004   2B    0.0    1 (ALIVE)               44
1       R_006   1A    0.0    1 (ALIVE)               34
2       R_013   1B    0.0    1 (ALIVE)               50
3       R_014   1A    0.0    1 (ALIVE)               25
4       R_019   1A    0.0     0 (DEAD)                3
5       R_020  N/A    NaN     0 (DEAD)                7
6       R_022   2B    0.0     0 (DEAD)               15
7       R_029  N/A    NaN     0 (DEAD)               10
8       R_033   2A    0.0     0 (DEAD)                9
9       R_035   3B    1.0    1 (ALIVE)               40
10      R_036   1B    0.0     0 (DEAD)               18
11      R_043   3A    1.0    1 (ALIVE)               24
12      R_053   2B    0.0     0 (DEAD)               28
13      R_056    4    1.0    1 (ALIVE)               46
14      R_061   1A    0.0    1 (ALIVE)               31
15      R_064   2B    0.0    1 (ALIVE)               53
16      R_065   3A    1.0    

## Cellule LungCT-7 — Extraction des features et sauvegarde HDF5

In [46]:
# ============================================================
# Extraction features LungCT-Diagnosis → HDF5
# ~58 patients × 75 slices = ~4350 slices
# Rapide — ~5 min sur GPU
# ============================================================
import os, h5py, gc
import numpy as np
import torch
import torch.nn as nn
import SimpleITK as sitk
import pandas as pd
from pathlib import Path
from docx import Document
from torchvision.models import resnet50, ResNet50_Weights

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {device}")

# --- Chemins ---
data_root  = Path('/kaggle/input/datasets/aubinyoumbi/lungct-diagnosis-tcia-mirrored-untampered')
lung_root  = data_root / 'LungCT-Diagnosis'
lung_dir   = lung_root / 'LungCT-Diagnosis'
OUTPUT_HDF5 = '/kaggle/working/lungct_features.hdf5'

HU_MIN, HU_MAX = -1000, 400
BATCH_SIZE     = 256

# --- Encoder ---
encoder = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
encoder.conv1 = nn.Conv2d(1, 64, kernel_size=7,
                           stride=2, padding=3, bias=False)
encoder.fc = nn.Identity()
encoder = encoder.to(device).eval()
print("Encoder chargé")

# --- Parser les labels depuis le docx ---
doc    = Document(lung_root /
    'LungCT-Diagnosis_SurvivalData_journal.pone_.0118261.s010.docx')
table3 = doc.tables[3]
rows   = [[c.text.strip() for c in row.cells] for row in table3.rows]
df_labels = pd.DataFrame(rows[1:],
    columns=['patient_id','tnm','vital_status','vital_dup',
             'survival_months','convexity','entropy_ratio'])
df_labels = df_labels[df_labels['patient_id'].str.startswith('Patient_')].copy()
df_labels['patient_id'] = df_labels['patient_id'].str.replace('Patient_','R_')

def tnm_to_label(tnm):
    if pd.isna(tnm) or tnm == 'N/A': return None
    if any(tnm.startswith(s) for s in ['1','2']): return 0  # précoce
    if any(tnm.startswith(s) for s in ['3','4']): return 1  # avancé
    return None

df_labels['label'] = df_labels['tnm'].apply(tnm_to_label)
label_dict = dict(zip(df_labels['patient_id'], df_labels['label']))
print(f"Labels chargés : {len(label_dict)} patients")

# --- Split train/test 80/20 reproductible ---
import hashlib
def to_split(pid, seed=42):
    h = int(hashlib.md5((pid+str(seed)).encode()).hexdigest(), 16)
    return 'test' if (h % 100) < 20 else 'train'

# --- Fonctions ---
def clip_and_normalize(arr_int16):
    p = arr_int16.astype(np.float32)
    p = np.clip(p, HU_MIN, HU_MAX)
    p = (p - HU_MIN) / (HU_MAX - HU_MIN)
    return p

def load_volume(patient_dir):
    """Charge le volume 3D complet d'un patient via DICOM series."""
    study  = sorted(patient_dir.iterdir())[0]
    series = sorted(study.iterdir())[0]
    reader = sitk.ImageSeriesReader()
    ids    = reader.GetGDCMSeriesIDs(str(series))
    names  = reader.GetGDCMSeriesFileNames(str(series), ids[0])
    reader.SetFileNames(names)
    vol = reader.Execute()
    return sitk.GetArrayFromImage(vol)  # (Z, H, W)

# --- Extraction ---
if os.path.exists(OUTPUT_HDF5):
    os.remove(OUTPUT_HDF5)
    print("Ancien HDF5 supprimé")

h5file = h5py.File(OUTPUT_HDF5, 'w')
n_done, n_skipped = 0, 0

# Patients valides : label connu + CT présent
valid_patients = [
    pid for pid, lbl in label_dict.items()
    if lbl is not None and (lung_dir / pid).exists()
]
print(f"Patients à traiter : {len(valid_patients)}")

for idx, pid in enumerate(sorted(valid_patients)):
    label  = label_dict[pid]
    split  = to_split(pid)
    p_dir  = lung_dir / pid

    # Charger le volume
    try:
        vol = load_volume(p_dir)          # (Z, H, W) int16
    except Exception as e:
        print(f"  SKIP {pid} : {e}")
        n_skipped += 1
        continue

    n_slices = vol.shape[0]

    # Normaliser toutes les slices
    slices_norm = clip_and_normalize(vol)  # (Z, H, W) float32

    # Redimensionner à 224×224 si nécessaire (ResNet attend ≥224)
    H, W = slices_norm.shape[1], slices_norm.shape[2]
    if H != 224 or W != 224:
        import torch.nn.functional as F
        t = torch.from_numpy(slices_norm).unsqueeze(1)  # (Z,1,H,W)
        t = F.interpolate(t, size=(224, 224), mode='bilinear',
                          align_corners=False)
        slices_tensor = t  # (Z, 1, 224, 224)
    else:
        slices_tensor = torch.from_numpy(slices_norm).unsqueeze(1)

    # Encoder par batch
    all_feats = []
    with torch.no_grad():
        for i in range(0, n_slices, BATCH_SIZE):
            batch = slices_tensor[i:i+BATCH_SIZE].to(device)
            feats = encoder(batch)              # (B, 2048)
            all_feats.append(feats.cpu().numpy())

    features_np = np.concatenate(all_feats, axis=0)  # (Z, 2048)

    # Écrire dans HDF5
    grp = h5file.create_group(pid)
    grp.create_dataset('img', data=features_np,
                       compression='gzip', compression_opts=6)
    grp.attrs['label'] = label
    grp.attrs['split'] = split
    grp.attrs['tnm']   = df_labels.loc[
        df_labels['patient_id']==pid, 'tnm'].values[0]
    grp.attrs['n_slices'] = n_slices

    del vol, slices_tensor, all_feats, features_np
    gc.collect()

    n_done += 1
    print(f"[{idx+1:2d}/{len(valid_patients)}] {pid} | "
          f"slices={n_slices} | label={label} | split={split}")

h5file.close()
print(f"\nTerminé — {n_done} patients, {n_skipped} ignorés")
print(f"HDF5 : {OUTPUT_HDF5}")
print(f"Taille : {os.path.getsize(OUTPUT_HDF5)/1e6:.1f} MB")

Device : cuda
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 184MB/s] 


Encoder chargé
Labels chargés : 62 patients
Patients à traiter : 61
[ 1/61] R_004 | slices=68 | label=0.0 | split=test
[ 2/61] R_006 | slices=130 | label=0.0 | split=train
[ 3/61] R_013 | slices=67 | label=0.0 | split=test
[ 4/61] R_014 | slices=76 | label=0.0 | split=train


ImageSeriesReader (0x55934620): Non uniform sampling or missing slices detected,  maximum nonuniformity:0.0983871



[ 5/61] R_019 | slices=63 | label=0.0 | split=train
[ 6/61] R_020 | slices=150 | label=nan | split=train
[ 7/61] R_022 | slices=80 | label=0.0 | split=train
[ 8/61] R_029 | slices=109 | label=nan | split=train
[ 9/61] R_033 | slices=67 | label=0.0 | split=test
[10/61] R_035 | slices=74 | label=1.0 | split=train
[11/61] R_036 | slices=65 | label=0.0 | split=train
[12/61] R_043 | slices=111 | label=1.0 | split=test
[13/61] R_053 | slices=104 | label=0.0 | split=train
[14/61] R_056 | slices=62 | label=1.0 | split=train
[15/61] R_061 | slices=70 | label=0.0 | split=test
[16/61] R_064 | slices=63 | label=0.0 | split=train
[17/61] R_065 | slices=69 | label=1.0 | split=train
[18/61] R_066 | slices=98 | label=1.0 | split=train
[19/61] R_069 | slices=118 | label=0.0 | split=test
[20/61] R_075 | slices=112 | label=0.0 | split=train
[21/61] R_077 | slices=49 | label=0.0 | split=test
[22/61] R_078 | slices=87 | label=0.0 | split=test
[23/61] R_093 | slices=65 | label=1.0 | split=test


ImageSeriesReader (0x57554410): Non uniform sampling or missing slices detected,  maximum nonuniformity:3.95402



[24/61] R_098 | slices=88 | label=1.0 | split=test
[25/61] R_102 | slices=55 | label=0.0 | split=train
[26/61] R_108 | slices=66 | label=1.0 | split=test
[27/61] R_111 | slices=83 | label=0.0 | split=train
[28/61] R_116 | slices=115 | label=0.0 | split=train
[29/61] R_117 | slices=78 | label=nan | split=train
[30/61] R_124 | slices=86 | label=0.0 | split=test
[31/61] R_126 | slices=71 | label=1.0 | split=test
[32/61] R_127 | slices=68 | label=1.0 | split=train
[33/61] R_137 | slices=69 | label=1.0 | split=train
[34/61] R_141 | slices=110 | label=0.0 | split=train
[35/61] R_143 | slices=65 | label=0.0 | split=train
[36/61] R_144 | slices=93 | label=0.0 | split=train
[37/61] R_146 | slices=87 | label=0.0 | split=train
[38/61] R_150 | slices=77 | label=0.0 | split=train
[39/61] R_157 | slices=34 | label=1.0 | split=train
[40/61] R_168 | slices=73 | label=1.0 | split=test
[41/61] R_170 | slices=63 | label=0.0 | split=train
[42/61] R_172 | slices=24 | label=0.0 | split=test
[43/61] R_175 | 

## Cellule LungCT-8 — Vérifier le HDF5 et préparer l'entraînement


In [47]:
import h5py
import numpy as np

h5 = h5py.File('/kaggle/working/lungct_features.hdf5', 'r')

train_pos, train_neg, train_nan = [], [], []
test_pos,  test_neg,  test_nan  = [], [], []

for uid in h5.keys():
    label = h5[uid].attrs['label']
    split = h5[uid].attrs['split']
    n     = h5[uid].attrs['n_slices']
    tnm   = h5[uid].attrs['tnm']

    if np.isnan(label):
        (train_nan if split=='train' else test_nan).append(uid)
        continue
    if label == 1:
        (train_pos if split=='train' else test_pos).append(uid)
    else:
        (train_neg if split=='train' else test_neg).append(uid)

print("=== Distribution après exclusion NaN ===")
print(f"Train — avancé(1) : {len(train_pos)}, précoce(0) : {len(train_neg)}, "
      f"total : {len(train_pos)+len(train_neg)}")
print(f"Test  — avancé(1) : {len(test_pos)},  précoce(0) : {len(test_neg)}, "
      f"total : {len(test_pos)+len(test_neg)}")
print(f"Exclus (NaN)      : train={len(train_nan)}, test={len(test_nan)}")

print(f"\nTrain positifs : {train_pos}")
print(f"Test  positifs : {test_pos}")

# Vérifier shapes
print(f"\n=== Shapes features ===")
for uid in list(h5.keys())[:3]:
    img = h5[uid]['img'][:]
    print(f"  {uid} : {img.shape}  label={h5[uid].attrs['label']}")

h5.close()

=== Distribution après exclusion NaN ===
Train — avancé(1) : 10, précoce(0) : 26, total : 36
Test  — avancé(1) : 7,  précoce(0) : 15, total : 22
Exclus (NaN)      : train=3, test=0

Train positifs : ['R_035', 'R_056', 'R_065', 'R_066', 'R_127', 'R_137', 'R_157', 'R_185', 'R_191', 'R_256']
Test  positifs : ['R_043', 'R_093', 'R_098', 'R_108', 'R_126', 'R_168', 'R_273']

=== Shapes features ===
  R_004 : (68, 2048)  label=0.0
  R_006 : (130, 2048)  label=0.0
  R_013 : (67, 2048)  label=0.0


## Cellule LungCT-9 — Dataset class

In [48]:
# data/lungct/lungct_dataset.py
import h5py
import numpy as np
import torch
from torch.utils.data import Dataset

class LungCTFeatures(Dataset):
    """
    Features pré-extraites LungCT-Diagnosis.
    Miroir exact de CamelyonFeatures / LUNAFeatures.
    Label : 0 = précoce (TNM I-II), 1 = avancé (TNM III-IV)
    """

    def open_hdf5(self):
        self.dataset = h5py.File(self.data_path, 'r')

    def select_scans(self):
        h5 = h5py.File(self.data_path, 'r')
        target_split = 'train' if self.train else 'test'
        self.scan_names = []
        for uid in h5.keys():
            label = h5[uid].attrs['label']
            split = h5[uid].attrs['split']
            # Exclure NaN labels
            if np.isnan(float(label)):
                continue
            if split == target_split:
                self.scan_names.append(uid)
        self.data_len = len(self.scan_names)
        h5.close()
        print(f"{'Train' if self.train else 'Test'} : "
              f"{self.data_len} patients")

    def __init__(self, conf, train=True):
        self.tasks     = conf.tasks
        self.data_path = conf.data_dir
        self.train     = train
        self.select_scans()

    def __len__(self):
        return self.data_len

    def __getitem__(self, i):
        if not hasattr(self, 'dataset'):
            self.open_hdf5()

        uid     = self.scan_names[i]
        scan    = self.dataset[uid]
        patches = scan['img'][:]          # (n_slices, 2048)
        label   = int(scan.attrs['label'])

        data_dict = {'input': torch.from_numpy(
                         patches.astype(np.float32))}
        for task in self.tasks.values():
            data_dict[task['name']] = label
        return data_dict

# Training

## training/iterative.py 

In [32]:
import sys
import numpy as np
import torch

# from utils.utils import adjust_learning_rate

def init_batch(device, conf):
    """
    Initialize the memory buffer for the batch consisting of M patches
    """
    if conf.is_image:
        mem_patch = torch.zeros((conf.B, conf.M, conf.n_chan_in, *conf.patch_size)).to(device)
    else:
        mem_patch = torch.zeros((conf.B, conf.M, conf.n_chan_in)).to(device)

    if conf.use_pos:
        mem_pos_enc = torch.zeros((conf.B, conf.M, conf.D)).to(device)
    else:
        mem_pos_enc = None

    # Init the labels for the batch (for multiple tasks in mnist)
    labels = {}
    for task in conf.tasks.values():
        if task['metric'] == 'multilabel_accuracy':
            labels[task['name']] = torch.zeros((conf.B, conf.n_class), dtype=torch.float32).to(device)
        else:
            labels[task['name']] = torch.zeros((conf.B,), dtype=torch.int64).to(device)
    
    return mem_patch, mem_pos_enc, labels

def fill_batch(mem_patch, mem_pos_enc, labels, data, n_prep, n_prep_batch,
            mem_patch_iter, mem_pos_enc_iter, conf):
    """
    Fill the patch, pos enc and label buffers and update helper variables
    """

    n_seq, len_seq = mem_patch_iter.shape[:2]
    mem_patch[n_prep:n_prep+n_seq, :len_seq] = mem_patch_iter
    if conf.use_pos:
        mem_pos_enc[n_prep:n_prep+n_seq, :len_seq] = mem_pos_enc_iter
    
    for task in conf.tasks.values():
        labels[task['name']][n_prep:n_prep+n_seq] = data[task['name']]
    
    n_prep += n_seq
    n_prep_batch += 1

    batch_data = (mem_patch, mem_pos_enc, labels, n_prep, n_prep_batch)

    return batch_data

def shrink_batch(mem_patch, mem_pos_enc, labels, n_prep, conf):
    """
    Adjust batch by removing empty instances (may occur in last batch of an epoch)
    """
    mem_patch = mem_patch[:n_prep]
    if conf.use_pos:
        mem_pos_enc = mem_pos_enc[:n_prep]
    
    for task in conf.tasks.values():
        labels[task['name']] = labels[task['name']][:n_prep]
    
    return mem_patch, mem_pos_enc, labels

def compute_loss(net, mem_patch, mem_pos_enc, criterions, labels, conf):
    """
    Obtain predictions, compute losses for each task and get some logging stats
    """

    # Obtain predictions
    preds = net(mem_patch, mem_pos_enc)

    # Compute losses for each task and sum them up
    loss = 0
    task_losses, task_preds, task_labels = {}, {}, {}
    for task in conf.tasks.values():
        t_name, t_act = task['name'], task['act_fn']

        criterion = criterions[t_name]
        label = labels[t_name]
        pred = preds[t_name].squeeze(-1)

        if t_act == 'softmax':
            pred_loss = torch.log(pred + conf.eps)
            label_loss = label
        else:
            pred_loss = pred.view(-1)
            label_loss = label.view(-1).type(torch.float32)

        task_loss = criterion(pred_loss, label_loss)
        # for logs
        task_losses[t_name] = task_loss.item()
        task_preds[t_name] = pred.detach().cpu().numpy()
        task_labels[t_name] = label.detach().cpu().numpy()

        loss += task_loss
    # Average task losses        
    loss /= len(conf.tasks.values())

    return loss, [task_losses, task_preds, task_labels]


def train_one_epoch(net, criterions, data_loader, optimizer, device, epoch, log_writer, conf):
    """
    Trains the given network for one epoch according to given criterions (loss functions)
    """

    # Set the network to training mode
    net.train()

    # Initialize helper variables
    n_prep, n_prep_batch = 0, 0 # num of prepared images/batches
    mem_pos_enc = None
    start_new_batch = True

    times = [] # only used when tracking efficiency stats
    # Loop through dataloader
    for data_it, data in enumerate(data_loader, start=epoch * len(data_loader)):
        # Move input batch onto GPU if eager execution is enabled (default), else leave it on CPU
        # Data is a dict with keys `input` (patches) and `{task_name}` (labels for given task)
        image_patches = data['input'].to(device) if conf.eager else data['input']

        # If starting a new batch, create placeholders for data which are filled later
        if start_new_batch:
            mem_patch, mem_pos_enc, labels = init_batch(device, conf)
            start_new_batch = False

            # If tracking efficiency, record time from here.
            if conf.track_efficiency:
                start_event = torch.cuda.Event(enable_timing=True)
                end_event = torch.cuda.Event(enable_timing=True)
                start_event.record()
        
        # Apply IPS to input patches
        mem_patch_iter, mem_pos_enc_iter = net.ips(image_patches)
        
        # Fill batch placeholders with patches from IPS step
        batch_data = fill_batch(mem_patch, mem_pos_enc, labels, data, n_prep, n_prep_batch,
                                mem_patch_iter, mem_pos_enc_iter, conf)
        mem_patch, mem_pos_enc, labels, n_prep, n_prep_batch = batch_data

        # Check if the current batch is full or if it is the last batch
        batch_full = (n_prep == conf.B)
        is_last_batch = n_prep_batch == len(data_loader)

        # Do training step as soon as batch is full (or last batch)
        if batch_full or is_last_batch:

            if not batch_full:
                # Last batch may not be full, so remove empty instances
                mem_patch, mem_pos_enc, labels = shrink_batch(mem_patch, mem_pos_enc, labels, n_prep, conf)
            
            # Calculate and set new learning rate
            adjust_learning_rate(conf.n_epoch_warmup, conf.n_epoch, conf.lr, optimizer, data_loader, data_it+1)
            optimizer.zero_grad()

            # Compute loss
            loss, task_info = compute_loss(net, mem_patch, mem_pos_enc, criterions, labels, conf)
            task_losses, task_preds, task_labels = task_info

            # Backpropagate error and update parameters
            loss.backward()
            optimizer.step()

            # If tracking efficiency, log the time and memory usage
            if conf.track_efficiency:
                end_event.record()
                torch.cuda.synchronize()
                if epoch == conf.track_epoch and data_it > 0 and not is_last_batch:
                    times.append(start_event.elapsed_time(end_event))
                    print("time: ", times[-1])

            # Update log
            log_writer.update(task_losses, task_preds, task_labels)

            # Reset helper variables
            n_prep = 0
            start_new_batch = True
    
    if conf.track_efficiency:
        if epoch == conf.track_epoch:
            print("avg. time: ", np.mean(times))

            stats = torch.cuda.memory_stats()
            peak_bytes_requirement = stats["allocated_bytes.all.peak"]
            print(f"Peak memory requirement: {peak_bytes_requirement / 1024 ** 3:.4f} GB")

            print("TORCH.CUDA.MEMORY_SUMMARY: ", torch.cuda.memory_summary())
            sys.exit()


# Disable gradient calculation during evaluation
@torch.no_grad()
def evaluate(net, criterions, data_loader, device, log_writer, conf):

    # Set the network to evaluation mode
    net.eval()

    # Remaining parts similar to training loop
    n_prep, n_prep_batch = 0, 0
    mem_pos_enc = None
    start_new_batch = True
    
    for data in data_loader:
        image_patches = data['input'].to(device) if conf.eager else data['input']

        if start_new_batch:
            mem_patch, mem_pos_enc, labels = init_batch(device, conf)
            start_new_batch = False
        
        mem_patch_iter, mem_pos_enc_iter = net.ips(image_patches)
        
        batch_data = fill_batch(mem_patch, mem_pos_enc, labels, data, n_prep, n_prep_batch,
                                mem_patch_iter, mem_pos_enc_iter, conf)
        mem_patch, mem_pos_enc, labels, n_prep, n_prep_batch = batch_data

        batch_full = (n_prep == conf.B)
        is_last_batch = n_prep_batch == len(data_loader)

        if batch_full or is_last_batch:

            if not batch_full:
                mem_patch, mem_pos_enc, labels = shrink_batch(mem_patch, mem_pos_enc, labels, n_prep, conf)
            
            _, task_info = compute_loss(net, mem_patch, mem_pos_enc, criterions, labels, conf)
            task_losses, task_preds, task_labels = task_info

            log_writer.update(task_losses, task_preds, task_labels)

            n_prep = 0
            start_new_batch = True

# Configs files

## camelyon_config.yml

In [33]:
import yaml

camelyon_config_yaml = """
#opt
n_epoch: 50           # number of epochs
B: 16                 # batch size
B_seq: 1              # sequential batch size, set either to
                      # B (eager and lazy loading) or 1 (eager sequential loading)
n_epoch_warmup: 10    # number of warm-up epochs
lr: 0.0003            # learning rate
wd: 0.1               # weight decay

#dset
n_class: 1                            # number of classes
data_dir: 'data/camelyon/dsets'       # directory of dataset
train_fname: 'feat_train_500ep.hdf5'  # filename of extracted features of training set
test_fname: 'feat_test_500ep.hdf5'    # filename of extracted features of test set
n_worker: 64                          # number of workers
pin_memory: False                     # use pin memory in dataloader
eager: True                           # eager or lazy loading

#misc
eps: 0.000001
seed: 0
track_efficiency: False   # for training, needs to be False
track_epoch: 0            # only relevant if efficiency stats are tracked.

#enc
is_image: False         # should a convolutional patch encoder be used?
enc_type: 'resnet50'    # used backbone, set either to 'resnet18' or 'resnet50'
pretrained: False       # should ImageNet weights be used?
n_chan_in: 2048         # number of input channels from resnet 50

#ips
shuffle: True             # should patches be shuffled?
shuffle_style: 'batch'    # 'batch' or 'instance'. 'batch' shuffles each instance of the batch the same way
n_token: 1                # number of learnable query tokens, corresponds to number of tasks
M: 5000                   # memory size
I: 5000                   # iteration size

#aggr
use_pos: False      # should positional encoding be used?
H: 8                # number of transformer layer heads
D: 512              # dimension of features
D_k: 64             # dimension of query/keys per head
D_v: 64             # dimension of values per head
D_inner: 2048       # intermediate layer dimension in MLP
attn_dropout: 0.1   # attention dropout
dropout: 0.1        # standard dropout

# define name, activation function of final layer and metric to be used
tasks:
  task0:
    id: 0
    name: 'metastases'
    act_fn: 'sigmoid'
    metric: 'auc'

"""

camelyon_conf = Struct(**yaml.safe_load(camelyon_config_yaml))

## mnist_config.yml

In [34]:
import yaml

mnist_config_yaml = """
#opt
n_epoch: 150          # number of epochs
B: 16                 # batch size
B_seq: 16             # sequential batch size, set either to
                      # B (eager and lazy loading) or 1 (eager sequential loading)
n_epoch_warmup: 10    # number of warm-up epochs
lr: 0.001             # learning rate
wd: 0.1               # weight decay

#dset
n_class: 10                                                   # number of classes
data_dir: '/kaggle/working/dsets/megapixel_mnist_1500'   # directory of dataset
n_worker: 4                                                   # number of workers
pin_memory: True                                              # use pin memory in dataloader
eager: True                                                   # eager or lazy loading

#misc
eps: 0.000001
seed: 0
track_efficiency: False   # for training, needs to be False
track_epoch: 0            # only relevant if efficiency stats are tracked.

#enc
is_image: True          # should a convolutional patch encoder be used?
enc_type: 'resnet18'    # used backbone, set either to 'resnet18' or 'resnet50'
pretrained: False       # should ImageNet weights be used?
n_chan_in: 1            # number of input channels
n_res_blocks: 2         # number of residual ResNet blocks, mnist only uses 2

#ips
shuffle: True               # should patches be shuffled?
shuffle_style: 'batch'      # 'batch' or 'instance'. 'batch' shuffles each instance of the batch the same way
n_token: 4                  # number of learnable query tokens, corresponds to number of tasks (mnist has 4 tasks)
N: 900                      # number of total patches, needs to be consistent with patch size/stride
M: 100                      # memory size
I: 100                      # iteration size
patch_size: [50, 50]        # dims of patch
patch_stride: [50, 50]      # stride of patch, use 25 per side for 50% overlap

#aggr
use_pos: True       # should positional encoding be used?
H: 8                # number of transformer layer heads
D: 128              # dimension of features
D_k: 16             # dimension of query/keys per head
D_v: 16             # dimension of values per head
D_inner: 512        # intermediate layer dimension in MLP
attn_dropout: 0.1   # attention dropout
dropout: 0.1        # standard dropout

# define name, activation fn of final layer and metric to be used
tasks:
  task0:
    id: 0
    name: 'majority'
    act_fn: 'softmax'
    metric: 'accuracy'
  task1:
    id: 1
    name: 'max'
    act_fn: 'softmax'
    metric: 'accuracy'
  task2:
    id: 2
    name: 'top'
    act_fn: 'softmax'
    metric: 'accuracy'
  task3:
    id: 3
    name: 'multi'
    act_fn: 'sigmoid'
    metric: 'multilabel_accuracy'


"""

mnist_conf = Struct(**yaml.safe_load(mnist_config_yaml))

## traffic_config.yml

In [35]:
import yaml

traffic_config_yaml = """
#opt
n_epoch: 150          # number of epochs
B: 16                 # batch size
B_seq: 16             # sequential batch size, set either to
                      # B (eager and lazy loading) or 1 (eager sequential loading)
n_epoch_warmup: 10    # number of warm-up epochs
lr: 0.0003            # learning rate
wd: 0.1               # weight decay

#dset
n_class: 4                      # number of classes
data_dir: 'data/traffic/dsets'  # directory of dataset
n_worker: 8                     # number of workers
pin_memory: True                # use pin memory in dataloader
eager: True                     # eager or lazy loading

#misc
eps: 0.000001
seed: 0
track_efficiency: False   # for training, needs to be False
track_epoch: 0            # only relevant if efficiency stats are tracked.

#enc
is_image: True          # should a convolutional patch encoder be used?
enc_type: 'resnet18'    # used backbone, set either to 'resnet18' or 'resnet50'
pretrained: True        # should ImageNet weights be used?
n_chan_in: 3            # number of input channels
n_res_blocks: 4         # number of residual ResNet blocks

#ips
shuffle: True                 # should patches be shuffled?
shuffle_style: 'batch'        # shuffle each instance the same way? 'batch' or 'instance'
n_token: 1                    # Number of learnable query tokens
N: 192                        # Number of total patches, needs to be consistent with patch size/stride
M: 10                         # memory size
I: 32                         # iteration size
patch_size: [100, 100]        # dims of patch
patch_stride: [100, 100]      # stride of patch

#aggr
use_pos: False        # should positional encoding be used?
H: 8                  # number of transformer layer heads
D: 512                # dimension of features
D_k: 64               # dimension of query/keys per head
D_v: 64               # dimension of values per head
D_inner: 2048         # hidden dimension of MLP
attn_dropout: 0.1     # attention dropout
dropout: 0.1          # standard dropout

tasks:
  task0:
    id: 0
    name: 'sign'
    act_fn: 'softmax'
    metric: 'accuracy'

"""

traffic_conf = Struct(**yaml.safe_load(traffic_config_yaml))

## Bloc B — Config LUNA16 :

In [36]:
import yaml

luna_config_yaml = """
#opt
n_epoch: 50
B: 16
B_seq: 1
n_epoch_warmup: 5
lr: 0.00005
wd: 0.5

#dset
n_class: 1
data_dir: '/kaggle/working/luna16_features.hdf5'
n_worker: 4
pin_memory: False
eager: True

#misc
eps: 0.000001
seed: 0
track_efficiency: False
track_epoch: 0

#enc
is_image: False        # features pré-extraites — projector linéaire utilisé
enc_type: 'resnet50'   # non utilisé quand is_image=False
pretrained: False      # non utilisé quand is_image=False
n_chan_in: 2048        # dimension features ResNet50 → input du projector
n_res_blocks: 4        # non utilisé quand is_image=False, mis pour cohérence

#ips
shuffle: True
shuffle_style: 'batch'
n_token: 1
M: 200                 # patches gardés — IPS sélectionne 200/621 (32%)
I: 200                 # patches traités par itération IPS
                       # n_iter = ceil((621-200)/200) = 3 itérations/scan


#aggr
use_pos: False         # pas de positional encoding
H: 4                   # têtes attention — réduit vs Camelyon (8) car dataset petit
D: 256                 # dim features — réduit vs Camelyon (512)
D_k: 32                # dim queries/keys par tête
D_v: 32                # dim values par tête
D_inner: 512           # dim MLP interne
attn_dropout: 0.3
dropout: 0.3

tasks:
  task0:
    id: 0
    name: 'nodule'
    act_fn: 'sigmoid'
    metric:
      - auc
      - f1
      - precision
      - recall
"""

luna_conf = Struct(**yaml.safe_load(luna_config_yaml))

luna_conf = Struct(**yaml.safe_load(luna_config_yaml))
print("Config LUNA16 chargée")
print(f"  data_dir   : {luna_conf.data_dir}")
print(f"  M={luna_conf.M}, I={luna_conf.I}, D={luna_conf.D}")
print(f"  n_chan_in   : {luna_conf.n_chan_in}")
print(f"  is_image    : {luna_conf.is_image}")

Config LUNA16 chargée
  data_dir   : /kaggle/working/luna16_features.hdf5
  M=200, I=200, D=256
  n_chan_in   : 2048
  is_image    : False


## Bloc B — Config NODE21 :

In [37]:
import yaml

node21_config_yaml = """
#opt
n_epoch: 50
B: 16                  # batch size
B_seq: 16              # eager loading — même que Traffic Signs
n_epoch_warmup: 5
lr: 0.0001
wd: 0.1

#dset
n_class: 2                                              # binaire : 0 ou 1
data_dir: '/kaggle/input/datasets/pshikk/node-21-dataset-untampered'
n_worker: 4
pin_memory: True
eager: True

#misc
eps: 0.000001
seed: 0
track_efficiency: False
track_epoch: 0

#enc
is_image: True         # patches RGB/gray directement
enc_type: 'resnet18'   # léger — images 1024×1024 avec patches 128×128
pretrained: True       # ImageNet weights
n_chan_in: 1           # grayscale
n_res_blocks: 4

#ips
shuffle: True
shuffle_style: 'batch'
n_token: 1
N: 64                  # 1024/128 × 1024/128 = 64 patches/image
M: 10                  # IPS sélectionne 10 parmi 64
I: 16                  # traite 16 patches par itération
patch_size: [128, 128] # 64 patches de 128×128 par image 1024×1024
patch_stride: [128, 128]

#aggr
use_pos: False
H: 8
D: 512
D_k: 64
D_v: 64
D_inner: 2048
attn_dropout: 0.1
dropout: 0.1

tasks:
  task0:
    id: 0
    name: 'nodule'
    act_fn: 'softmax'
    metric:
      - accuracy
      - auc
      - f1
      - precision
      - recall
"""

node21_conf = Struct(**yaml.safe_load(node21_config_yaml))
print("Config NODE21 chargée")
print(f"  patch_size : {node21_conf.patch_size}")
print(f"  N={node21_conf.N}, M={node21_conf.M}, I={node21_conf.I}")
print(f"  n_chan_in  : {node21_conf.n_chan_in}")

Config NODE21 chargée
  patch_size : [128, 128]
  N=64, M=10, I=16
  n_chan_in  : 1


## Cellule LungCT-10 — Config

In [49]:
import yaml

lungct_config_yaml = """
#opt
n_epoch: 50
B: 8                   # petit dataset — batch réduit
B_seq: 1               # un patient à la fois
n_epoch_warmup: 5
lr: 0.00005            # lr faible — 58 patients seulement
wd: 0.5                # régularisation forte

#dset
n_class: 1
data_dir: '/kaggle/working/lungct_features.hdf5'
n_worker: 2
pin_memory: False
eager: True

#misc
eps: 0.000001
seed: 0
track_efficiency: False
track_epoch: 0

#enc
is_image: False        # features pré-extraites
enc_type: 'resnet50'   # non utilisé
pretrained: False      # non utilisé
n_chan_in: 2048        # dim features ResNet50
n_res_blocks: 4        # non utilisé

#ips
shuffle: True
shuffle_style: 'batch'
n_token: 1
M: 30                  # sélectionne 30 parmi ~75 slices (40%)
I: 30                  # traite 30 par itération

#aggr
use_pos: False
H: 2                   # très petit — 58 patients
D: 128                 # très petit
D_k: 16
D_v: 16
D_inner: 256
attn_dropout: 0.3
dropout: 0.3

tasks:
  task0:
    id: 0
    name: 'stage'
    act_fn: 'sigmoid'
    metric:
      - auc
      - f1
      - precision
      - recall
"""

lungct_conf = Struct(**yaml.safe_load(lungct_config_yaml))
print("Config LungCT chargée")
print(f"  M={lungct_conf.M}, I={lungct_conf.I}, "
      f"D={lungct_conf.D}, H={lungct_conf.H}")
print(f"  n_chan_in={lungct_conf.n_chan_in}, "
      f"is_image={lungct_conf.is_image}")

Config LungCT chargée
  M=30, I=30, D=128, H=2
  n_chan_in=2048, is_image=False


# Section 7 — Training runner 

## main

In [50]:
#!/usr/bin/env python

import os
import yaml
from pprint import pprint

import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader

# from utils.utils import Logger, Struct
# from data.megapixel_mnist.mnist_dataset import MegapixelMNIST
# from data.traffic.traffic_dataset import TrafficSigns
# from data.camelyon.camelyon_dataset import CamelyonFeatures
# from architecture.ips_net import IPSNet
# from training.iterative import train_one_epoch, evaluate

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

dataset = 'lungct' # either one of {'mnist', 'camelyon', 'traffic', 'luna', 'node21', 'lungct'}

conf_map = {'mnist': mnist_conf, 'traffic': traffic_conf, 'camelyon': camelyon_conf, 'luna': luna_conf, 'node21':  node21_conf, 'lungct': lungct_conf
 }
conf = conf_map[dataset]
pprint(vars(conf))

# fix the seed for reproducibility
torch.manual_seed(conf.seed)
np.random.seed(conf.seed)

# define datasets and dataloaders
if dataset == 'mnist':
    train_data = MegapixelMNIST(conf, train=True)
    test_data = MegapixelMNIST(conf, train=False)
elif dataset == 'traffic':
    train_data = TrafficSigns(conf, train=True)
    test_data = TrafficSigns(conf, train=False)
elif dataset == 'camelyon':
    train_data = CamelyonFeatures(conf, train=True)
    test_data = CamelyonFeatures(conf, train=False)
elif dataset == 'luna':
    train_data = LUNAFeatures(luna_conf, train=True)
    test_data  = LUNAFeatures(luna_conf, train=False)
elif dataset == 'node21':
    train_data = NODE21Dataset(node21_conf, train=True)
    test_data  = NODE21Dataset(node21_conf, train=False)
elif dataset == 'lungct':
    train_data = LungCTFeatures(lungct_conf, train=True)
    test_data  = LungCTFeatures(lungct_conf, train=False)

    # WeightedRandomSampler pour équilibrer 10 pos / 26 neg
    from torch.utils.data import WeightedRandomSampler
    h5tmp = h5py.File(lungct_conf.data_dir, 'r')
    weights = []
    for uid in train_data.scan_names:
        lbl = int(h5tmp[uid].attrs['label'])
        weights.append(2.6 if lbl == 1 else 1.0)
    h5tmp.close()
    sampler = WeightedRandomSampler(
        weights=weights,
        num_samples=len(weights),
        replacement=True
    )
    train_loader = DataLoader(
        train_data, batch_size=lungct_conf.B_seq,
        sampler=sampler,              # ← remplace shuffle=True
        num_workers=lungct_conf.n_worker,
        pin_memory=lungct_conf.pin_memory,
        persistent_workers=True
    )
    # test_loader reste inchangé
    test_loader = DataLoader(
        test_data, batch_size=lungct_conf.B_seq,
        shuffle=False,
        num_workers=lungct_conf.n_worker,
        pin_memory=lungct_conf.pin_memory,
        persistent_workers=True
    )



train_loader = DataLoader(train_data, batch_size=conf.B_seq, shuffle=True,
    num_workers=conf.n_worker, pin_memory=conf.pin_memory, persistent_workers=True)
test_loader = DataLoader(test_data, batch_size=conf.B_seq, shuffle=False,
    num_workers=conf.n_worker, pin_memory=conf.pin_memory, persistent_workers=True)

# define network
net = IPSNet(device, conf).to(device)

loss_nll = nn.NLLLoss()
loss_bce = nn.BCELoss()

# define optimizer, lr not important at this point
optimizer = torch.optim.AdamW(net.parameters(), lr=0, weight_decay=conf.wd)

criterions = {}
for task in conf.tasks.values():
    criterions[task['name']] = loss_nll if task['act_fn'] == 'softmax' else loss_bce

log_writer_train = Logger(conf.tasks)
log_writer_test = Logger(conf.tasks)

for epoch in range(conf.n_epoch):
    
    train_one_epoch(net, criterions, train_loader, optimizer, device, epoch, log_writer_train, conf)

    log_writer_train.compute_metric()

    more_to_print = {'lr': optimizer.param_groups[0]['lr']}
    log_writer_train.print_stats(epoch, train=True, **more_to_print)

    evaluate(net, criterions, test_loader, device, log_writer_test, conf)
    
    log_writer_test.compute_metric()
    log_writer_test.print_stats(epoch, train=False)

{'B': 8,
 'B_seq': 1,
 'D': 128,
 'D_inner': 256,
 'D_k': 16,
 'D_v': 16,
 'H': 2,
 'I': 30,
 'M': 30,
 'attn_dropout': 0.3,
 'data_dir': '/kaggle/working/lungct_features.hdf5',
 'dropout': 0.3,
 'eager': True,
 'enc_type': 'resnet50',
 'eps': 1e-06,
 'is_image': False,
 'lr': 5e-05,
 'n_chan_in': 2048,
 'n_class': 1,
 'n_epoch': 50,
 'n_epoch_warmup': 5,
 'n_res_blocks': 4,
 'n_token': 1,
 'n_worker': 2,
 'pin_memory': False,
 'pretrained': False,
 'seed': 0,
 'shuffle': True,
 'shuffle_style': 'batch',
 'tasks': {'task0': {'act_fn': 'sigmoid',
                     'id': 0,
                     'metric': ['auc', 'f1', 'precision', 'recall'],
                     'name': 'stage'}},
 'track_efficiency': False,
 'track_epoch': 0,
 'use_pos': False,
 'wd': 0.5}
Train : 36 patients
Test : 22 patients
Train Epoch: 1
task: stage, loss: 0.68716, auc: 0.53846, f1: 0.32000, precision: 0.26667, recall: 0.40000
avg loss: 0.68716, lr: 1e-05

Test Epoch: 1
task: stage, loss: 0.70499, auc: 0.29524, 